<a href="https://colab.research.google.com/github/tijanicica/ai-speak/blob/main/AVATAR_trifon_srpskiwav_knjige_5speakera_NODISC_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# 1. PODEŠAVANJE SISTEMA I UVOZ BIBLIOTEKA
# ==============================================================================

import os
import gc
import math
import time
import random
import warnings
from typing import Optional, Dict, List, Tuple

# PyTorch moduli za arhitekturu modela, optimizaciju i rad sa podacima
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import OneCycleLR

# Numerička analiza i upravljanje strukturiranim podacima
import numpy as np
import pandas as pd

# Digitalna obrada audio signala i ekstrakcija karakteristika
import librosa
import torchaudio
import torchaudio.transforms as T
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model

# Obrada signala, interpolacija podataka i proračun korelacije
from scipy import signal
from scipy.interpolate import interp1d
from scipy.stats import pearsonr

# Moduli za vizuelizaciju rezultata
import matplotlib.pyplot as plt
import seaborn as sns

# Onemogućavanje sistemskih upozorenja (warnings)
warnings.filterwarnings('ignore')

def set_seed(seed: int = 42):
    """
    Fiksiranje svih generatora nasumičnih brojeva kako bi rezultati
    bili identični pri svakom ponovnom pokretanju koda.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Inicijalizacija seed-a i postavljanje uređaja za rad (CPU/GPU)
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Provera dostupnosti grafičke kartice i stanja resursa
if torch.cuda.is_available():
    print(f"Status: CUDA aktivan | GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Resursi: {vram:.2f} GB dostupne VRAM memorije")
else:
    print("Status: CUDA drajver nije detektovan. Izvršavanje na CPU-u.")

In [ ]:
# ==========================================
# 2. DATASET I OBRADA PODATAKA
# ==========================================

# Srpski skup fonema
SERBIAN_PHONEMES = [
    'a', 'b', 'v', 'g', 'd', 'đ', 'e', 'ž', 'z', 'i', 'j', 'k', 'l', 'lj', 'm',
    'n', 'nj', 'o', 'p', 'r', 's', 't', 'ć', 'u', 'f', 'h', 'c', 'č', 'dž', 'š',
    'sil'
]

PHONEME_TO_IDX = {p: i for i, p in enumerate(SERBIAN_PHONEMES)}
NUM_PHONEMES = len(SERBIAN_PHONEMES)

# Nazivi blendshape-ova (ARKit 52)
BLENDSHAPE_NAMES = [
    'browInnerUp', 'browDownLeft', 'browDownRight', 'browOuterUpLeft', 'browOuterUpRight',
    'eyeLookUpLeft', 'eyeLookUpRight', 'eyeLookDownLeft', 'eyeLookDownRight',
    'eyeLookInLeft', 'eyeLookInRight', 'eyeLookOutLeft', 'eyeLookOutRight',
    'eyeBlinkLeft', 'eyeBlinkRight', 'eyeSquintLeft', 'eyeSquintRight',
    'eyeWideLeft', 'eyeWideRight', 'cheekPuff', 'cheekSquintLeft', 'cheekSquintRight',
    'noseSneerLeft', 'noseSneerRight', 'jawOpen', 'jawForward', 'jawLeft', 'jawRight',
    'mouthFunnel', 'mouthPucker', 'mouthLeft', 'mouthRight', 'mouthRollUpper',
    'mouthRollLower', 'mouthShrugUpper', 'mouthShrugLower', 'mouthClose',
    'mouthSmileLeft', 'mouthSmileRight', 'mouthFrownLeft', 'mouthFrownRight',
    'mouthDimpleLeft', 'mouthDimpleRight', 'mouthUpperUpLeft', 'mouthUpperUpRight',
    'mouthLowerDownLeft', 'mouthLowerDownRight', 'mouthPressLeft', 'mouthPressRight',
    'mouthStretchLeft', 'mouthStretchRight', 'tongueOut'
]

# Koliko frejmova unapred model sme da "vidi" (200ms lookahead)
LOOKAHEAD_MS = 200
LOOKAHEAD_FRAMES_AUDIO = int(LOOKAHEAD_MS / 1000 * 50)  # Wav2Vec2 radi na 50Hz
LOOKAHEAD_FRAMES_BS    = int(LOOKAHEAD_MS / 1000 * 60)  # Blendshape-ovi su na 60Hz

class AdvancedAudioProcessor:
    def __init__(self, model_name="/content/drive/MyDrive/AI-SPEAK/srpski_wav2vec/model_0815_2ep_srKnjige", device='cuda'):
        self.device = device
        self.feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
        self.model = Wav2Vec2Model.from_pretrained(model_name).to(device)
        self.model.eval()
        for param in self.model.parameters():
            param.requires_grad = False  # zamrzavamo težine, ne treniramo Wav2Vec2

    def extract_features(self, waveform):
        waveform = waveform / (waveform.abs().max() + 1e-8)  # normalizacija amplitude
        with torch.no_grad():
            inputs = self.feature_extractor(
                waveform.squeeze().cpu().numpy(),
                sampling_rate=16000,
                return_tensors="pt",
                padding=True
            )
            outputs = self.model(
                inputs.input_values.to(self.device),
                output_hidden_states=True
            )
            # Uzimamo prosek poslednja 4 skrivena sloja za bogatiju reprezentaciju zvuka
            hidden_states = outputs.hidden_states[-4:]
            features = torch.stack(hidden_states).mean(0).squeeze(0)

        features = (features - features.mean(dim=-1, keepdim=True)) / (features.std(dim=-1, keepdim=True) + 1e-8)

        return features.cpu()

class AggressiveDenoiser:
    # Savitzky-Golay filter za glatke pokrete lica - uklanja sitne trzaje
    def __init__(self):
        self.regions = {
            'jaw_mouth': slice(24, 52),
            'upper_face': slice(0, 24)
        }

    def denoise(self, blendshapes):
        denoised = blendshapes.copy()
        # Gornji deo lica (oči, obrve) - jači filter jer ti pokreti treba da budu glatki
        denoised[:, self.regions['upper_face']] = signal.savgol_filter(blendshapes[:, self.regions['upper_face']], 11, 2, axis=0)
        # Usta i vilica - slabiji filter da se ne izgubi oštrina govora
        denoised[:, self.regions['jaw_mouth']] = signal.savgol_filter(blendshapes[:, self.regions['jaw_mouth']], 5, 2, axis=0)
        return np.clip(denoised, 0, 1)

class ChampionshipDataset(Dataset):
    def __init__(self, base_path, speakers=['spk08', 'spk14'], split='train', fps=60, augment=True, device='cuda', cache_dir="audio_cache"):
        self.base_path = base_path
        self.speakers = speakers
        self.fps = fps
        self.augment = augment and split == 'train'
        self.device = device
        self.cache_dir = cache_dir  # folder gde čuvamo već izvučene audio features

        if not os.path.exists(self.cache_dir):
            os.makedirs(self.cache_dir, exist_ok=True)

        # Prolazimo kroz sve govornike i skupljamo validne parove (audio + blendshape + foneme)
        self.samples = []
        for speaker in self.speakers:
            is_new = speaker in ['spk03', 'spk04', 'spk05']
            spk_num = speaker.replace('spk', '')

            if is_new:
                audio_dir = os.path.join(base_path, f'SPEAKER_{spk_num:0>2}',
                                        f'SPEAKER_{spk_num:0>2}_blendshapes_and_audio')
            else:
                audio_dir = os.path.join(base_path, f'{speaker}_blendshapes', f'renamed_{speaker}')

            if not os.path.exists(audio_dir): continue

            if is_new:
                csv_files = [f for f in os.listdir(audio_dir) if f.startswith(f'{speaker}_') and f.endswith('.csv')]
                f_ids = [f.replace(f'{speaker}_', '').replace('.csv', '') for f in csv_files]
            else:
                csv_files = [f for f in os.listdir(audio_dir) if f.endswith('.csv')]
                f_ids = [f.replace(f'{speaker}_', '').replace('.csv', '') for f in csv_files]

            for f_id in f_ids:
                if is_new:
                    wav_file = os.path.join(audio_dir, f'FC_{f_id}.wav')
                    csv_file = os.path.join(audio_dir, f'{speaker}_{f_id}.csv')
                else:
                    wav_file = os.path.join(audio_dir, f'{speaker}_{f_id}.wav')
                    csv_file = os.path.join(audio_dir, f'{speaker}_{f_id}.csv')

                ph_file = os.path.join(base_path, 'labels_aligned', 'labels_aligned',
                                      'per_phoneme', f'{speaker}_{f_id}.txt')

                if os.path.exists(wav_file) and os.path.exists(csv_file) and os.path.exists(ph_file):
                    self.samples.append((speaker, f_id))


        SHARED_IDS = {f'{i:03d}' for i in range(1, 31)}  # 001-030 zajedničke
        SHARED_TRAIN = {f'{i:03d}' for i in range(1, 21)}  # 001-020 train
        SHARED_VAL   = {f'{i:03d}' for i in range(21, 31)} # 021-030 val

        # NOVO — ista logika za sve 5 govornika:
        old_shared_train = [(spk, fid) for spk, fid in self.samples
                            if spk in ['spk08', 'spk14'] and fid in SHARED_TRAIN]
        old_shared_val   = [(spk, fid) for spk, fid in self.samples
                            if spk in ['spk08', 'spk14'] and fid in SHARED_VAL]
        old_unique       = [(spk, fid) for spk, fid in self.samples
                            if spk in ['spk08', 'spk14'] and fid not in SHARED_IDS]

        new_shared_train = [(spk, fid) for spk, fid in self.samples
                            if spk in ['spk03','spk04','spk05'] and fid in SHARED_TRAIN]
        new_shared_val   = [(spk, fid) for spk, fid in self.samples
                            if spk in ['spk03','spk04','spk05'] and fid in SHARED_VAL]
        new_unique       = [(spk, fid) for spk, fid in self.samples
                            if spk in ['spk03','spk04','spk05'] and fid not in SHARED_IDS]

        random.seed(42)
        random.shuffle(old_unique)
        split_idx = int(len(old_unique) * 0.80)
        old_unique_train = old_unique[:split_idx]
        old_unique_val   = old_unique[split_idx:]

        random.seed(42)
        random.shuffle(new_unique)
        split_idx2 = int(len(new_unique) * 0.80)
        new_unique_train = new_unique[:split_idx2]
        new_unique_val   = new_unique[split_idx2:]

        if split == 'train':
            self.samples = old_shared_train + old_unique_train + new_shared_train + new_unique_train
        else:
            self.samples = old_shared_val + old_unique_val + new_shared_val + new_unique_val




        # Audio procesor inicijalizujemo tek kad zatreba (lazy loading)
        self.audio_processor = None
        self.denoiser = AggressiveDenoiser()

        print(f"{split.upper()} SET: {len(self.samples)} sentences.")

    def _load_triphone_indices(self, ph_path, num_frames):
        # Učitavamo trenutne fonemske indekse kao i pre
        phoneme_indices = np.zeros(num_frames, dtype=np.int64)
        with open(ph_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 3:
                    start, end, ph = float(parts[0]), float(parts[1]), parts[2].lower()
                    if ph in PHONEME_TO_IDX:
                        s_frame = int(start * self.fps)
                        e_frame = int(end * self.fps)
                        phoneme_indices[max(0, s_frame):min(num_frames, e_frame)] = PHONEME_TO_IDX[ph]

        # Prethodni fonem — pomeramo za jedan frejm (prvi frejm kopiramo)
        prev_indices = np.concatenate([[phoneme_indices[0]], phoneme_indices[:-1]])

        # Naredni fonem — pomeramo za jedan frejm (poslednji frejm kopiramo)
        next_indices = np.concatenate([phoneme_indices[1:], [phoneme_indices[-1]]])

        return prev_indices, phoneme_indices, next_indices



    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        speaker, file_id = self.samples[idx]
        cache_path = os.path.join(self.cache_dir, f"{speaker}_{file_id}.pt")

        # Putanje do svih potrebnih fajlova
        is_new = speaker in ['spk03', 'spk04', 'spk05']
        spk_num = speaker.replace('spk', '')
        if is_new:
            base_dir   = os.path.join(self.base_path, f'SPEAKER_{spk_num:0>2}',
                                      f'SPEAKER_{spk_num:0>2}_blendshapes_and_audio')
            audio_path = os.path.join(base_dir, f'FC_{file_id}.wav')
            bs_path    = os.path.join(base_dir, f'{speaker}_{file_id}.csv')
        else:
            base_dir   = os.path.join(self.base_path, f'{speaker}_blendshapes', f'renamed_{speaker}')
            audio_path = os.path.join(base_dir, f'{speaker}_{file_id}.wav')
            bs_path    = os.path.join(base_dir, f'{speaker}_{file_id}.csv')

        ph_path = os.path.join(self.base_path, 'labels_aligned', 'labels_aligned',
                              'per_phoneme', f'{speaker}_{file_id}.txt')

        cache_path = os.path.join(self.cache_dir, f"{speaker}_{file_id}.pt")

        # 1. Učitavanje audio featura - ako postoji keš koristimo njega, inače ekstrahujemo
        if os.path.exists(cache_path):
            audio_features = torch.load(cache_path, map_location='cpu')
        else:
            # Tek sad palimo težak Wav2Vec2 model jer nema keširanog rezultata
            if self.audio_processor is None:
                self.audio_processor = AdvancedAudioProcessor(device=self.device)
            waveform, _ = librosa.load(audio_path, sr=16000)
            audio_features = self.audio_processor.extract_features(torch.from_numpy(waveform).unsqueeze(0))
            torch.save(audio_features, cache_path)  # snimamo da ne računamo ponovo

        # 2. Učitavanje blendshape vrednosti iz CSV-a
        df = pd.read_csv(bs_path, header=None)
        blendshapes = df.values.astype(np.float32)[:, :52]
        num_frames = blendshapes.shape[0]

        # 3. Usklađivanje dužina - Wav2Vec2 je na 50Hz a blendshape-ovi na 60Hz
        audio_len = audio_features.shape[0]
        if audio_len != num_frames:
            interp = interp1d(np.linspace(0, 1, audio_len), audio_features.numpy(), axis=0, kind='linear')
            audio_features = torch.from_numpy(interp(np.linspace(0, 1, num_frames))).float()

        prev_ph, curr_ph, next_ph = self._load_triphone_indices(ph_path, num_frames)


        # 4. Računanje energije signala po frejmu
        waveform_raw, _ = librosa.load(audio_path, sr=16000)
        energy = librosa.feature.rms(y=waveform_raw, hop_length=int(16000/self.fps))[0]
        energy = np.pad(energy, (0, max(0, num_frames - len(energy))))[:num_frames]

        speaker_id_map = {'spk08': 0, 'spk14': 1, 'spk03': 2, 'spk04': 3, 'spk05': 4}
        speaker_id = speaker_id_map.get(speaker, 0)


        return {
            'audio_features': audio_features,
            'prev_phoneme': torch.from_numpy(prev_ph).long(),
            'curr_phoneme': torch.from_numpy(curr_ph).long(),
            'next_phoneme': torch.from_numpy(next_ph).long(),
            'blendshapes': torch.from_numpy(self.denoiser.denoise(blendshapes)),
            'energy': torch.from_numpy(energy.astype(np.float32)),
            'speaker_id': torch.tensor(speaker_id, dtype=torch.long)
        }

def create_dataloaders(base_path, speakers=['spk08', 'spk14'], batch_size=32, num_workers=4, device='cuda', cache_dir="audio_cache"):
    # Prosleđujemo cache_dir datasetu
    train_dataset = ChampionshipDataset(base_path, speakers=speakers, split='train', device=device, cache_dir=cache_dir)
    val_dataset = ChampionshipDataset(base_path, speakers=speakers, split='val', device=device, cache_dir=cache_dir)

    def collate_fn(batch):
        # Padujemo sve sekvence na dužinu najduže u batchu
        max_len = max([item['audio_features'].shape[0] for item in batch])

        audio_padded, prev_ph_padded, curr_ph_padded, next_ph_padded, bs_padded, en_padded, masks = [], [], [], [], [], [], []
        speaker_ids = torch.stack([item['speaker_id'] for item in batch])

        for item in batch:
            seq_len = item['audio_features'].shape[0]
            pad_len = max_len - seq_len

            audio_padded.append(F.pad(item['audio_features'], (0, 0, 0, pad_len)))
            prev_ph_padded.append(F.pad(item['prev_phoneme'], (0, pad_len)))
            curr_ph_padded.append(F.pad(item['curr_phoneme'], (0, pad_len)))
            next_ph_padded.append(F.pad(item['next_phoneme'], (0, pad_len)))
            bs_padded.append(F.pad(item['blendshapes'], (0, 0, 0, pad_len)))
            en_padded.append(F.pad(item['energy'], (0, pad_len)))

            # Maska koja govori modelu koji frejmovi su pravi a koji padding
            mask = torch.zeros(max_len, dtype=torch.bool)
            mask[:seq_len] = True
            masks.append(mask)

        return {
            'audio_features': torch.stack(audio_padded),
            'prev_phoneme': torch.stack(prev_ph_padded),
            'curr_phoneme': torch.stack(curr_ph_padded),
            'next_phoneme': torch.stack(next_ph_padded),
            'blendshapes': torch.stack(bs_padded),
            'energy': torch.stack(en_padded),
            'speaker_ids': speaker_ids,
            'mask': torch.stack(masks)
        }

    # num_workers i pin_memory za brže učitavanje podataka
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=num_workers,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=num_workers,
        pin_memory=True
    )

    return train_loader, val_loader

print("Dataset je spreman")

In [ ]:
# ==============================================================================
# 3. POVEZIVANJE SA GOOGLE DRIVE-OM
# ==============================================================================

from google.colab import drive

# Povezivanje Drive-a radi pristupa podacima i čuvanja rezultata treninga
drive.mount('/content/drive')

# Definisanje putanje do podataka na Google Drive-u
BASE_DATA_PATH = "/content/drive/MyDrive/AI-SPEAK"

In [ ]:
# ==============================================================================
# DATASET STATISTICS (Section 2.1)
# ==============================================================================
import pandas as pd

train_ds = ChampionshipDataset(BASE_DATA_PATH,
    speakers=['spk08','spk14','spk03','spk04','spk05'],
    split='train',
    cache_dir='/content/drive/MyDrive/AI-SPEAK/cache_normalized')

val_ds = ChampionshipDataset(BASE_DATA_PATH,
    speakers=['spk08','spk14','spk03','spk04','spk05'],
    split='val',
    cache_dir='/content/drive/MyDrive/AI-SPEAK/cache_normalized')

rows = []
for spk in ['spk08','spk14','spk03','spk04','spk05']:
    tr = [(s,f) for s,f in train_ds.samples if s == spk]
    va = [(s,f) for s,f in val_ds.samples if s == spk]
    rows.append({'Speaker': spk, 'Train': len(tr), 'Val': len(va), 'Total': len(tr)+len(va)})

df_stats = pd.DataFrame(rows)
df_stats.loc[len(df_stats)] = ['TOTAL', df_stats.Train.sum(), df_stats.Val.sum(), df_stats.Total.sum()]
print(df_stats.to_string(index=False))

In [ ]:
# ==============================================================================
# 4. IZDVAJANJE I ČUVANJE AUDIO KARAKTERISTIKA (CACHING)
# ==============================================================================

from tqdm.auto import tqdm
import gc

def pre_cache_all_audio(base_path, speakers=['spk08', 'spk14'], cache_dir="audio_cache", device='cuda'):
    print("Starting audio feature extraction and caching...")

    train_samples = ChampionshipDataset(base_path, speakers=speakers, split='train', cache_dir=cache_dir).samples
    val_samples = ChampionshipDataset(base_path, speakers=speakers, split='val', cache_dir=cache_dir).samples
    all_samples = train_samples + val_samples

    processor = AdvancedAudioProcessor(device=device)

    for speaker, file_id in tqdm(all_samples, desc="Ekstrakcija"):
        cache_path = os.path.join(cache_dir, f"{speaker}_{file_id}.pt")
        if not os.path.exists(cache_path):
            is_new = speaker in ['spk03', 'spk04', 'spk05']
            spk_num = speaker.replace('spk', '')
            if is_new:
                audio_dir  = os.path.join(base_path, f'SPEAKER_{spk_num:0>2}',
                                          f'SPEAKER_{spk_num:0>2}_blendshapes_and_audio')
                audio_path = os.path.join(audio_dir, f'FC_{file_id}.wav')
            else:
                audio_path = os.path.join(base_path, f'{speaker}_blendshapes', f'renamed_{speaker}',
                                          f'{speaker}_{file_id}.wav')
            try:
                waveform, _ = librosa.load(audio_path, sr=16000)
                features = processor.extract_features(torch.from_numpy(waveform).unsqueeze(0))
                torch.save(features, cache_path)
            except Exception as e:
                print(f"Error on file {file_id}: {e}")

    del processor
    torch.cuda.empty_cache()
    gc.collect()
    print("Caching complete. VRAM memory released.")



# Provera putanje i pokretanje funkcije
if os.path.exists(BASE_DATA_PATH):
    print("Path is valid. Starting process.")
    pre_cache_all_audio(
        BASE_DATA_PATH,
        speakers=['spk08', 'spk14', 'spk03', 'spk04', 'spk05'],
        cache_dir="/content/drive/MyDrive/AI-SPEAK/cache_normalized"
    )
else:
    print("Error: Data path not found. Check Drive connection.")

In [ ]:
# ==============================================================================
# 5. ARHITEKTURA MODELA I DISKRIMINATORA
# ==============================================================================

class VectorQuantizer(nn.Module):
    """
    Sloj koji služi za Vector Quantization. Definisan je ovde,
    ali ga u forward delu trenutno preskačemo.
    """
    def __init__(self, num_embeddings=512, embedding_dim=256, commitment_cost=0.25):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.commitment_cost = commitment_cost

        self.embeddings = nn.Embedding(num_embeddings, embedding_dim)
        self.embeddings.weight.data.uniform_(-1/num_embeddings, 1/num_embeddings)

    def forward(self, z):
        B, T, D = z.shape
        z_flattened = z.reshape(-1, D)

        distances = (
            torch.sum(z_flattened**2, dim=1, keepdim=True)
            + torch.sum(self.embeddings.weight**2, dim=1)
            - 2 * torch.matmul(z_flattened, self.embeddings.weight.t())
        )

        encoding_indices = torch.argmin(distances, dim=1)
        encodings = F.one_hot(encoding_indices, self.num_embeddings).float()

        quantized = torch.matmul(encodings, self.embeddings.weight)
        quantized = quantized.view(B, T, D)

        e_latent_loss = F.mse_loss(quantized.detach(), z)
        q_latent_loss = F.mse_loss(quantized, z.detach())
        codebook_loss = q_latent_loss + self.commitment_cost * e_latent_loss

        quantized = z + (quantized - z).detach()

        return quantized, codebook_loss, encoding_indices.view(B, T)


class PositionalEncoding(nn.Module):
    """
    Dodajemo informaciju o tome koji je frejm po redu u vremenu,
    kako bi model znao redosled govora.
    """
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class ChampionshipLipSyncModel(nn.Module):
    """
    Glavni model koji spaja audio, foneme i ID govornika
    i pretvara ih u pokrete lica.
    """
    def __init__(
        self,
        audio_dim=1024,
        phoneme_embedding_dim=16,
        d_model=512,
        num_encoder_layers=4,
        num_heads=8,
        dim_feedforward=2048,
        num_blendshapes=52,
        codebook_size=512,
        gru_hidden=512,
        dropout=0.1
    ):
        super().__init__()

        self.d_model = d_model
        self.gru_hidden = gru_hidden

        self.prev_phoneme_embedding = nn.Embedding(NUM_PHONEMES, phoneme_embedding_dim)  # 16
        self.curr_phoneme_embedding = nn.Embedding(NUM_PHONEMES, phoneme_embedding_dim)  # 16
        self.next_phoneme_embedding = nn.Embedding(NUM_PHONEMES, phoneme_embedding_dim)  # 16
        # 1024 + 16 + 16 + 16 = 1072
        self.input_projection = nn.Linear(audio_dim + phoneme_embedding_dim * 3, d_model)

        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_encoder_layers)

        self.pre_quant = nn.Linear(d_model, 256)
        self.vq = VectorQuantizer(num_embeddings=codebook_size, embedding_dim=256)
        self.post_quant = nn.Linear(256, d_model)

        self.gru = nn.GRU(
            input_size=d_model,
            hidden_size=gru_hidden,
            num_layers=2,
            batch_first=True,
            dropout=dropout if num_encoder_layers > 1 else 0
        )

        self.jaw_head = self._make_head(gru_hidden, 4)
        self.mouth_head = self._make_head(gru_hidden, 24)
        self.eye_head = self._make_head(gru_hidden, 14)
        self.brow_head = self._make_head(gru_hidden, 5)
        self.other_head = self._make_head(gru_hidden, 5)

        nn.init.constant_(self.brow_head[-2].bias, -0.3)

    def _make_head(self, input_dim, num_outputs):
        head = nn.Sequential(
            nn.Linear(input_dim, input_dim // 2),
            nn.LayerNorm(input_dim // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(input_dim // 2, num_outputs),
            nn.Sigmoid()
        )
        nn.init.constant_(head[-2].bias, -3.0)
        return head

    def forward(self, audio_features, prev_phoneme, curr_phoneme, next_phoneme, speaker_ids=None, mask=None, hidden_state=None):
        B, T, _ = audio_features.shape
        prev_emb = self.prev_phoneme_embedding(prev_phoneme)
        curr_emb = self.curr_phoneme_embedding(curr_phoneme)
        next_emb = self.next_phoneme_embedding(next_phoneme)
        x = torch.cat([audio_features, prev_emb, curr_emb, next_emb], dim=-1)
        x = self.input_projection(x)
        x = self.pos_encoder(x)

        if mask is not None:
            src_key_padding_mask = ~mask
        else:
            src_key_padding_mask = None

        encoded = self.encoder(x, mask=None, src_key_padding_mask=src_key_padding_mask)


        # Trenutno preskačemo VQ i šaljemo nulu za vq_loss
        quantized = encoded
        vq_loss = torch.tensor(0.0).to(x.device)

        if hidden_state is None:
            gru_out, hidden_state = self.gru(quantized)
        else:
            gru_out, hidden_state = self.gru(quantized, hidden_state)

        jaw = self.jaw_head(gru_out)
        mouth = self.mouth_head(gru_out)
        eye = self.eye_head(gru_out)
        brow = self.brow_head(gru_out)
        other = self.other_head(gru_out)

        blendshapes = torch.cat([brow, eye, other, jaw, mouth], dim=-1)

        return blendshapes, vq_loss, hidden_state


class MultiScaleDiscriminator(nn.Module):
    """
    Diskriminator koji proverava koliko pokreti izgledaju realno
    u različitim vremenskim razmacima.
    """
    def __init__(self, num_blendshapes=52):
        super().__init__()

        self.disc_1x = self._make_discriminator(num_blendshapes, scale=1)
        self.disc_2x = self._make_discriminator(num_blendshapes, scale=2)
        self.disc_4x = self._make_discriminator(num_blendshapes, scale=4)

    def _make_discriminator(self, in_channels, scale):
        return nn.ModuleDict({
            'downsample': nn.AvgPool1d(scale, scale) if scale > 1 else nn.Identity(),
            'conv_blocks': nn.Sequential(
                nn.Conv1d(in_channels, 128, kernel_size=15, padding=7),
                nn.LeakyReLU(0.2),
                nn.Conv1d(128, 256, kernel_size=11, stride=2, padding=5),
                nn.LeakyReLU(0.2),
                nn.Conv1d(256, 512, kernel_size=7, stride=2, padding=3),
                nn.LeakyReLU(0.2),
                nn.Conv1d(512, 512, kernel_size=5, stride=2, padding=2),
                nn.LeakyReLU(0.2),
            ),
            'classifier': nn.Conv1d(512, 1, kernel_size=3, padding=1)
        })

    def _forward_single_scale(self, x, disc):
        x = x.transpose(1, 2)
        x = disc['downsample'](x)
        features = disc['conv_blocks'](x)
        score = disc['classifier'](features)
        return score, features

    def forward(self, blendshapes):
        scores = []
        features = []

        for disc in [self.disc_1x, self.disc_2x, self.disc_4x]:
            s, f = self._forward_single_scale(blendshapes, disc)
            scores.append(s)
            features.append(f)

        return scores, features

print("Model architecture loaded.")

In [ ]:
# ==============================================================================
# ARCHITECTURE DIAGRAM (Section 3.2)
# ==============================================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

fig, ax = plt.subplots(figsize=(14, 5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 5)
ax.axis('off')

boxes = [
    (0.3, 1.5, 1.8, 2.0, 'Audio\n(wav)', '#AED6F1'),
    (0.3, 3.5, 1.8, 2.0, 'Phoneme\n(triphone)', '#A9DFBF'),
    (2.5, 2.0, 2.0, 2.0, 'Wav2Vec2\n(frozen)', '#AED6F1'),
    (2.5, 3.5, 2.0, 2.0, 'Phoneme\nEmbedding\n(3×16)', '#A9DFBF'),
    (5.0, 2.5, 2.0, 2.0, 'Linear\nProjection\n(1072→512)', '#FAD7A0'),
    (7.2, 2.5, 2.0, 2.0, 'Transformer\nEncoder\n(4 layers)', '#F9E79F'),
    (9.4, 2.5, 2.0, 2.0, 'GRU\n(2 layers)', '#D2B4DE'),
    (11.6, 2.5, 2.0, 2.0, 'Multi-head\nOutput\n(52 BS)', '#ABEBC6'),
]

for x, y, w, h, label, color in boxes:
    rect = mpatches.FancyBboxPatch((x, y), w, h,
        boxstyle="round,pad=0.1", facecolor=color, edgecolor='gray', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center', fontsize=8.5, fontweight='bold')

arrows = [
    (2.1, 2.5, 2.5, 3.0),
    (2.1, 4.5, 2.5, 4.5),
    (4.5, 3.0, 5.0, 3.5),
    (4.5, 4.5, 5.0, 3.8),
    (7.0, 3.5, 7.2, 3.5),
    (9.2, 3.5, 9.4, 3.5),
    (11.4, 3.5, 11.6, 3.5),
]

for x1, y1, x2, y2 in arrows:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
        arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

ax.set_title('Model Architecture: Audio-Driven Facial Animation for Serbian',
             fontsize=12, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig('architecture_diagram.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================================================
# 6. FUNKCIJE GUBITKA I KRITERIJUMI OPTIMIZACIJE
# ==============================================================================

def correlation_loss(pred, target, mask=None):
    """
    Računanje Pearsonove korelacije za praćenje trendova kroz vreme.
    """
    pred_mean = pred.mean(dim=1, keepdim=True)
    target_mean = target.mean(dim=1, keepdim=True)

    pred_centered = pred - pred_mean
    target_centered = target - target_mean

    numerator = (pred_centered * target_centered).sum(dim=1)
    denominator = torch.sqrt(
        (pred_centered ** 2).sum(dim=1) * (target_centered ** 2).sum(dim=1) + 1e-8
    )

    # Ograničavanje vrednosti na opseg [-1, 1] radi numeričke stabilnosti
    corr = torch.clamp(numerator / denominator, -1.0, 1.0)
    return 1.0 - corr.mean()

def velocity_correlation_loss(pred, target, mask=None):
    """
    Korelacija brzine pokreta radi postizanja bolje dinamike.
    """
    if pred.shape[1] < 2:
        return torch.tensor(0.0).to(pred.device)
    pred_vel = pred[:, 1:] - pred[:, :-1]
    target_vel = target[:, 1:] - target[:, :-1]
    mask_vel = mask[:, 1:] if mask is not None else None
    return correlation_loss(pred_vel, target_vel, mask_vel)

class ChampionshipLoss(nn.Module):
    """
    Glavna klasa za gubitak koja kombinuje rekonstrukciju, korelaciju,
    perceptualne metrike i glatkoću pokreta.
    """
    def __init__(
        self,
        reconstruction_weight=42.0,
        correlation_weight=185.0,
        velocity_weight=140.0,
        perceptual_weight=0.30,
        smoothness_weight=1e-9,
        amplitude_weight=180.0,
        jaw_amplitude_weight=150.0
    ):
        super().__init__()
        self.reconstruction_weight = reconstruction_weight
        self.correlation_weight = correlation_weight
        self.velocity_weight = velocity_weight
        self.perceptual_weight = perceptual_weight
        self.smoothness_weight = smoothness_weight
        self.amplitude_weight = amplitude_weight
        self.jaw_amplitude_weight = jaw_amplitude_weight

        # Indeksi ključnih blendshape grupa
        self.mouth_indices = list(range(28, 52))
        self.jaw_idx = 24
        self.close_idx = 36

    def reconstruction_loss(self, pred, target, energy=None, mask=None):
        """
        L1 rekonstrukcija sa dinamičkim pojačivačima za brzinu i amplitudu.
        """
        l1 = torch.abs(pred - target)

        # Brzina promene ciljnih podataka
        target_vel = torch.zeros_like(target)
        target_vel[:, 1:] = torch.abs(target[:, 1:] - target[:, :-1])

        # Dinamička pojačanja za oštriju artikulaciju
        velocity_boost = 1.0 + target_vel * 50.0
        amplitude_boost = 1.0 + target * 15.0

        # Težine po regijama lica (vilica i usta imaju prioritet)
        weights = torch.ones_like(l1)
        weights[:, :, self.mouth_indices] *= 80.0
        weights[:, :, self.jaw_idx] *= 115.0
        weights[:, :, 0:5] *= 75.0    # Brows
        weights[:, :, 5:19] *= 45.0   # Eyes
        weights[:, :, 19:24] *= 2.0   # Other

        # Kazna za neželjene pokrete tokom tišine (Silence penalty)
        silence_mask = (target < 0.01).float()
        silence_penalty = (silence_mask * pred * 60.0)

        loss = (l1 * weights * amplitude_boost * velocity_boost) + silence_penalty

        # Povećanje kretanja obrva na osnovu jačine zvuka
        if energy is not None:
            brow_energy_boost = energy.unsqueeze(-1) * 35.0
            loss[:, :, 0:5] += (l1[:, :, 0:5] * brow_energy_boost)

        if mask is not None:
            loss = loss * mask.unsqueeze(-1)
            return loss.sum() / (mask.sum() * pred.shape[-1] + 1e-8)
        return loss.mean()

    def correlation_loss_wrapper(self, pred, target, mask=None):
        """
        Računanje korelacije po kanalima uz definisane težine značaja.
        """
        total_corr_loss = torch.tensor(0.0, device=pred.device)
        weight_sum = 0

        for i in range(pred.shape[-1]):
            pred_i = pred[:, :, i]
            target_i = target[:, :, i]
            if target_i.std() < 0.001: continue

            curr_corr = correlation_loss(pred_i, target_i, mask)

            # Diferencirane težine za tajming pokreta
            if i in range(0, 5): w = 25.0       # Obrve su važne za emociju
            elif i == self.jaw_idx: w = 15.0
            elif i in [28, 29, 36]: w = 12.0
            elif i in self.mouth_indices: w = 5.0
            else: w = 1.0

            total_corr_loss = total_corr_loss + (curr_corr * w)
            weight_sum += w
        return total_corr_loss / (weight_sum + 1e-8)

    def velocity_loss_func(self, pred, target, mask=None):
        """
        Kombinacija intenziteta i korelacije brzine pokreta.
        """
        if pred.shape[1] < 2: return torch.tensor(0.0).to(pred.device)

        pred_vel = pred[:, 1:] - pred[:, :-1]
        target_vel = target[:, 1:] - target[:, :-1]

        vel_mag_loss = torch.abs(pred_vel - target_vel).mean() * 300.0
        vel_corr_loss = velocity_correlation_loss(pred[:, :, self.jaw_idx], target[:, :, self.jaw_idx], mask)

        return vel_mag_loss + vel_corr_loss * 20.0

    def peak_amplitude_loss(self, pred, target, mask=None):
        """
        Osigurava da model dostiže pune amplitude u ključnim momentima.
        """
        p_max, _ = torch.max(pred[:, :, [self.jaw_idx, 28, 29, 36]], dim=1)
        t_max, _ = torch.max(target[:, :, [self.jaw_idx, 28, 29, 36]], dim=1)
        diff = torch.clamp(t_max - p_max, min=0)
        return (diff ** 2).mean() * 60.0

    def jaw_amplitude_loss_func(self, pred, target, mask=None):
        """
        Gubitak za maksimalno otvaranje vilice.
        """
        p_jaw = pred[:, :, self.jaw_idx]
        t_jaw = target[:, :, self.jaw_idx]
        return F.mse_loss(p_jaw.max(dim=1)[0], t_jaw.max(dim=1)[0])

    def perceptual_loss(self, pred_features, target_features):
        """
        Gubitak zasnovan na unutrašnjim karakteristikama diskriminatora.
        """
        if not pred_features:
            return torch.tensor(0.0, device='cuda')

        total_loss = torch.tensor(0.0, device=pred_features[0].device)
        for pred_f, target_f in zip(pred_features, target_features):
            total_loss = total_loss + F.l1_loss(pred_f, target_f)
        return total_loss / (len(pred_features) + 1e-8)

    def smoothness_loss(self, pred, mask=None):
        """
        Kažnjavanje naglih ubrzanja radi dobijanja prirodnijih pokreta.
        """
        accel = pred[:, 2:] - 2*pred[:, 1:-1] + pred[:, :-2]

        weights = torch.ones_like(accel)
        weights[:, :, 0:5]   *= 3.0   # Maksimalna glatkoća za obrve
        weights[:, :, 5:19]  *= 1.5
        weights[:, :, 24:28] *= 2.0
        weights[:, :, 28:52] *= 1.2   # Očuvanje oštrine govora

        if mask is not None:
            loss = (accel.pow(2) * weights * mask[:, 2:].unsqueeze(-1)).sum()
            return loss / (mask[:, 2:].sum() * pred.shape[-1] + 1e-8)
        return (accel.pow(2) * weights).mean()

    def forward(self, pred, target, pred_features, target_features, energy=None, mask=None):
        """
        Suma svih komponenti u finalni gubitak.
        """
        recon = self.reconstruction_loss(pred, target, energy, mask)
        corr = self.correlation_loss_wrapper(pred, target, mask)
        vel = self.velocity_loss_func(pred, target, mask)
        perc = self.perceptual_loss(pred_features, target_features)
        smooth = self.smoothness_loss(pred, mask)
        amp = self.peak_amplitude_loss(pred, target, mask)
        jaw_amp = self.jaw_amplitude_loss_func(pred, target, mask)

        # Poseban fokus na preciznost zatvaranja usta
        close_target = target[:, :, self.close_idx]
        close_pred = pred[:, :, self.close_idx]
        close_impact = F.mse_loss(close_pred * (close_target > 0.6).float(),
                                  close_target * (close_target > 0.6).float())

        total = (
            self.reconstruction_weight * recon +
            self.correlation_weight * corr +
            self.velocity_weight * vel +
            self.perceptual_weight * perc +
            self.smoothness_weight * smooth +
            self.amplitude_weight * amp +
            self.jaw_amplitude_weight * jaw_amp +
            60.0 * close_impact
        )

        mouth_recon = F.l1_loss(pred[:, :, self.mouth_indices], target[:, :, self.mouth_indices])

        return {
            'total': total,
            'reconstruction': recon,
            'mouth_recon': mouth_recon,
            'correlation': corr,
            'velocity': vel,
            'perceptual': perc,
            'smoothness': smooth,
            'amplitude': amp,
            'jaw_amplitude': jaw_amp,
            'jaw_velocity': vel
        }

print("All loss functions defined.")

In [ ]:
# ==============================================================================
# 7. LOGIKA TRENERA I PROCES OBUČAVANJA
# ==============================================================================

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR
import numpy as np
from tqdm.auto import tqdm
from scipy.stats import pearsonr

class ChampionshipTrainer:
    """
    Glavna klasa za upravljanje treningom. Brine o optimizaciji generatora
    i diskriminatora, Mixed Precision režimu i praćenju metrika.
    """
    def __init__(
        self,
        model,
        discriminator,
        train_loader,
        val_loader,
        device='cuda',
        learning_rate=1e-4,
        gradient_accumulation_steps=2,
        use_wandb=False
    ):
        self.model = model.to(device)
        self.discriminator = discriminator.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.gradient_accumulation_steps = gradient_accumulation_steps
        self.use_wandb = use_wandb
        self.history = {'train_corr': [], 'val_corr': [], 'val_loss': [], 'train_loss': []}


        # Podešavanje svih komponenti funkcije gubitka
        self.criterion = ChampionshipLoss(
            reconstruction_weight=42.0,
            correlation_weight=185.0,
            velocity_weight=140.0,
            amplitude_weight=180.0,
            jaw_amplitude_weight=170.0,
            perceptual_weight=0.30,
            smoothness_weight=8e-4
        )
        self.adversarial_loss = nn.MSELoss()

        # AdamW optimizatori sa težinskim opadanjem
        self.optimizer_g = optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.5, 0.999), weight_decay=1e-4)

        total_steps = len(train_loader) // gradient_accumulation_steps * 200

        # OneCycleLR za stabilnije i brže učenje
        self.scheduler_g = OneCycleLR(self.optimizer_g, max_lr=learning_rate, total_steps=total_steps, pct_start=0.15, anneal_strategy='cos', div_factor=20.0, final_div_factor=100.0)

        # Skaliranje gradijenata za stabilniji rad u Mixed Precision modu
        self.scaler_g = torch.cuda.amp.GradScaler()

        self.best_val_loss = float('inf')
        self.best_val_corr = 0.0
        self.patience = 80
        self.patience_counter = 0

    def reset_gru_hidden(self, batch_size):
        """Resetovanje skrivenog stanja GRU mreže na početku svakog batch-a."""
        return None

    def train_discriminator(self, batch, accumulation_step):
        """Učenje diskriminatora da prepozna autentične pokrete lica."""
        audio = batch['audio_features'].to(self.device)
        prev_ph = batch['prev_phoneme'].to(self.device)
        curr_ph = batch['curr_phoneme'].to(self.device)
        next_ph = batch['next_phoneme'].to(self.device)
        target = batch['blendshapes'].to(self.device)
        mask = batch['mask'].to(self.device)

        with torch.cuda.amp.autocast():
            with torch.no_grad():
                hidden = self.reset_gru_hidden(audio.shape[0])
                fake, vq_loss, hidden = self.model(audio, prev_ph, curr_ph, next_ph, mask=mask, hidden_state=hidden)

            real_scores, _ = self.discriminator(target)
            fake_scores, _ = self.discriminator(fake.detach())

            d_loss_real = sum([F.mse_loss(s, torch.ones_like(s)) for s in real_scores]) / len(real_scores)
            d_loss_fake = sum([F.mse_loss(s, torch.zeros_like(s)) for s in fake_scores]) / len(fake_scores)
            d_loss = (d_loss_real + d_loss_fake) / (2 * self.gradient_accumulation_steps)

        self.scaler_d.scale(d_loss).backward()

        if (accumulation_step + 1) % self.gradient_accumulation_steps == 0:
            self.scaler_d.step(self.optimizer_d)
            self.scaler_d.update()
            self.optimizer_d.zero_grad()
            self.scheduler_d.step()

        return d_loss.item() * self.gradient_accumulation_steps

    def train_generator(self, batch, accumulation_step, epoch):
        """Učenje generatora da stvara realne pokrete uz adversarial feedback."""
        audio = batch['audio_features'].to(self.device)
        prev_ph = batch['prev_phoneme'].to(self.device)
        curr_ph = batch['curr_phoneme'].to(self.device)
        next_ph = batch['next_phoneme'].to(self.device)
        target = batch['blendshapes'].to(self.device)
        energy = batch['energy'].to(self.device)
        mask = batch['mask'].to(self.device)

        with torch.cuda.amp.autocast():
            hidden = self.reset_gru_hidden(audio.shape[0])
            fake, vq_loss, hidden = self.model(audio, prev_ph, curr_ph, next_ph, mask=mask, hidden_state=hidden)

            losses = self.criterion(fake, target, [], [], energy, mask)
            g_loss = (losses['total'] + 1.0 * vq_loss) / self.gradient_accumulation_steps

        self.scaler_g.scale(g_loss).backward()

        if (accumulation_step + 1) % self.gradient_accumulation_steps == 0:
            self.scaler_g.unscale_(self.optimizer_g)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.5)
            self.scaler_g.step(self.optimizer_g)
            self.scaler_g.update()
            self.optimizer_g.zero_grad()
            self.scheduler_g.step()

        return {
            'g_loss': g_loss.item() * self.gradient_accumulation_steps,
            'reconstruction': losses['reconstruction'].item(),
            'mouth_recon': losses['mouth_recon'].item(),
            'correlation': losses['correlation'].item(),
            'velocity': losses['velocity'].item(),
            'perceptual': losses['perceptual'].item(),
            'vq_loss': vq_loss.item() if isinstance(vq_loss, torch.Tensor) else vq_loss,
            'amplitude': losses['amplitude'].item(),
            'jaw_amp': losses['jaw_amplitude'].item(),
            'jaw_vel': losses['jaw_velocity'].item()
        }

    @torch.no_grad()
    def validate(self):
        """Provera modela na podacima koje nije video tokom učenja."""
        self.model.eval()
        val_losses, val_corrs = [], []

        for batch in self.val_loader:
            audio = batch['audio_features'].to(self.device)
            prev_ph = batch['prev_phoneme'].to(self.device)
            curr_ph = batch['curr_phoneme'].to(self.device)
            next_ph = batch['next_phoneme'].to(self.device)
            target = batch['blendshapes'].to(self.device)
            energy = batch['energy'].to(self.device)
            mask = batch['mask'].to(self.device)

            with torch.cuda.amp.autocast():
                hidden = self.reset_gru_hidden(audio.shape[0])
                fake, vq_loss, hidden = self.model(audio, prev_ph, curr_ph, next_ph, mask=mask, hidden_state=hidden)
                losses = self.criterion(fake, target, [], [], energy, mask)


            val_losses.append(losses['total'].item())
            pred_np, target_np = fake[mask].cpu().numpy(), target[mask].cpu().numpy()

            if len(pred_np) > 10:
                corr, _ = pearsonr(pred_np.flatten(), target_np.flatten())
                if not np.isnan(corr): val_corrs.append(corr)

        self.model.train()
        return np.mean(val_losses), np.mean(val_corrs) if val_corrs else 0.0

    def train(self, num_epochs, save_dir='checkpoints', resume_from=None):
        """Glavna trening petlja sa vizuelnim praćenjem napretka."""
        os.makedirs(save_dir, exist_ok=True)
        training_start_time = time.time()

        start_epoch = 0

        # Nastavak treninga od checkpoint-a ako je prosleđen
        if resume_from and os.path.exists(resume_from):
            print(f"Nastavljam trening od: {resume_from}")
            ckpt = torch.load(resume_from, map_location=self.device, weights_only=False)
            self.model.load_state_dict(ckpt['model_state_dict'])
            self.optimizer_g.load_state_dict(ckpt['optimizer_g_state_dict'])
            self.best_val_corr = ckpt.get('val_correlation', 0.0)
            start_epoch = ckpt.get('epoch', 0) + 1
            print(f"Nastavljam od epohe {start_epoch}, best corr: {self.best_val_corr:.4f}")

        self.optimizer_g.zero_grad()

        for epoch in range(start_epoch, num_epochs):
            self.model.train()
            epoch_metrics = {k: [] for k in ['g_loss', 'reconstruction', 'mouth_recon', 'correlation', 'velocity', 'perceptual', 'vq_loss', 'd_loss', 'amplitude', 'jaw_amp', 'jaw_vel']}

            pbar = tqdm(self.train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')

            for batch_idx, batch in enumerate(pbar):
                g_metrics = self.train_generator(batch, batch_idx, epoch)
                for k, v in g_metrics.items():
                    if k in epoch_metrics: epoch_metrics[k].append(v)

                pbar.set_postfix({
                    'G': f"{np.mean(epoch_metrics['g_loss']):.2f}",
                    'Corr': f"{np.mean(epoch_metrics['correlation']):.3f}",
                    'Rec': f"{np.mean(epoch_metrics['reconstruction']):.3f}",
                    'Mouth': f"{np.mean(epoch_metrics['mouth_recon']):.3f}",
                    'Jaw': f"{np.mean(epoch_metrics['jaw_amp']):.3f}",
                    'Best': f"{self.best_val_corr:.3f}"
                })

            val_loss, val_corr = self.validate()
            self.history['val_corr'].append(val_corr)
            self.history['val_loss'].append(val_loss)
            self.history['train_corr'].append(np.mean(epoch_metrics['correlation']))
            self.history['train_loss'].append(np.mean(epoch_metrics['g_loss']))
            print(f"Epoch {epoch+1} - Val Loss: {val_loss:.4f}, Val Correlation: {val_corr:.4f}")

            # Checkpoint koji se uvek čuva nakon svake epohe (za resume)
            last_checkpoint = {
                'epoch': epoch,
                'model_state_dict': self.model.state_dict(),
                'optimizer_g_state_dict': self.optimizer_g.state_dict(),
                'val_correlation': val_corr,
                'val_loss': val_loss
            }


            torch.save(last_checkpoint, os.path.join(save_dir, 'last_epoch.pt'))

            # Best model checkpoint
            if val_corr > self.best_val_corr:
                self.best_val_corr = val_corr
                self.best_val_loss = val_loss
                self.patience_counter = 0
                torch.save(last_checkpoint, os.path.join(save_dir, 'best_model.pt'))
                print(f"Model improved and saved (Corr: {val_corr:.4f})")
            else:
                self.patience_counter += 1

            if self.patience_counter >= self.patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

            torch.cuda.empty_cache()

        total_training_time = time.time() - training_start_time
        hours = int(total_training_time // 3600)
        minutes = int((total_training_time % 3600) // 60)
        seconds = int(total_training_time % 60)
        self.history['total_training_time_sec'] = total_training_time
        self.history['epochs_trained'] = epoch + 1

        print(f"Training complete. Best correlation: {self.best_val_corr:.4f}")
        print(f"Total training time: {hours}h {minutes}m {seconds}s")
        print(f"Epochs trained: {epoch + 1}")
        print(f"Average time per epoch: {total_training_time / (epoch + 1):.1f}s")

In [ ]:
# ==============================================================================
# 8. KONFIGURACIJA I POKRETANJE PROCESA OBUČAVANJA
# ==============================================================================

# Podešavanje za efikasnije korišćenje grafičke memorije
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

def train_championship_model(
    base_path='/content/drive/MyDrive/AI-SPEAK',
    speakers=['spk08', 'spk14', 'spk03', 'spk04', 'spk05'],
    save_dir='/content/drive/MyDrive/AVATAR_CHECKPOINTS_5SPK_NODISCR',  # NOVI FOLDER
    resume_from=None,

    # Podešavanja za stabilnost i rad sa memorijom
    batch_size=8,
    gradient_accumulation_steps=8, # Efektivni batch size je 64 (8x8)

    learning_rate=1e-4,
    num_epochs=200,
    d_model=512,
    num_encoder_layers=4,
    codebook_size=512,
    use_wandb=False,
    device='cuda'
):
    print("-" * 70)
    print("TRAINING SYSTEM - MEMORY OPTIMIZATION ENABLED")
    print("-" * 70)
    print(f"Effective Batch Size: {batch_size * gradient_accumulation_steps}")

    # Pravljenje foldera za čuvanje modela ako već ne postoji
    if not os.path.exists(save_dir):
        os.makedirs(save_dir, exist_ok=True)

    # Pražnjenje memorije pre starta
    torch.cuda.empty_cache()
    gc.collect()

    print("Loading datasets...")
    train_loader, val_loader = create_dataloaders(
        base_path,
        speakers=speakers,
        batch_size=batch_size,
        num_workers=0, # Postavljeno na 0 radi veće stabilnosti u okruženju
        device=device,
        cache_dir="/content/drive/MyDrive/AI-SPEAK/cache_normalized"
    )

    print("Building model and discriminator...")
    model = ChampionshipLipSyncModel(
        audio_dim=1024,
        d_model=d_model,
        num_encoder_layers=num_encoder_layers,
        codebook_size=codebook_size
    )
    discriminator = MultiScaleDiscriminator(num_blendshapes=52)

    print("Initializing trainer...")
    trainer = ChampionshipTrainer(
        model=model,
        discriminator=discriminator,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        learning_rate=learning_rate,
        gradient_accumulation_steps=gradient_accumulation_steps,
        use_wandb=use_wandb
    )

    print(f"\nStarting training (Batch: {batch_size}, Accumulation: {gradient_accumulation_steps})\n")


    try:
        trainer.train(num_epochs=num_epochs, save_dir=save_dir, resume_from=resume_from)
    except KeyboardInterrupt:
        print("\nTraining interrupted. Saving current weights...")
        torch.save(model.state_dict(), os.path.join(save_dir, 'interrupted_model.pt'))

    return trainer


if __name__ == "__main__":
    # Povezivanje na Drive pre pokretanja procesa
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    trainer = train_championship_model()

In [ ]:
# ==============================================================================
# 9. INFERENCIJA I POST-PROCESIRANJE REZULTATA
# ==============================================================================

from scipy.signal import savgol_filter

class MinimalPostProcessor:
    """
    Sređuje pokrete da budu simetrični i da nema nerealno naglih skokova u animaciji.
    """
    def __init__(self, fps=60):
        self.fps = fps
        self.constraints = {
            'eye_symmetry': {'pairs': [(13, 14)], 'max_diff': 0.3},
            'jaw_mutual_exclusive': {'indices': [24, 36], 'max_sum': 1.2},
            'max_velocity': {
                'jaw': 0.9, 'mouth': 0.8, 'eyes': 0.50, 'default': 0.45
            }
        }

    def enforce_symmetry(self, blendshapes):
        """Sređivanje simetrije očiju."""
        processed = blendshapes.copy()
        for pair in self.constraints['eye_symmetry']['pairs']:
            left_idx, right_idx = pair
            max_diff = self.constraints['eye_symmetry']['max_diff']
            diff = np.abs(processed[:, left_idx] - processed[:, right_idx])
            excessive = diff > max_diff
            if excessive.any():
                avg = (processed[excessive, left_idx] + processed[excessive, right_idx]) / 2
                processed[excessive, left_idx] = avg
                processed[excessive, right_idx] = avg
        return processed

    def clamp_extreme_velocity(self, blendshapes):
        """Ograničavanje brzine kretanja da se izbegne trzanje."""
        processed = blendshapes.copy()
        jaw_indices = [24, 25, 26, 27]
        mouth_indices = list(range(28, 52))
        eye_indices = list(range(5, 19))

        def clamp_group(indices, max_vel):
            for idx in indices:
                velocity = np.diff(processed[:, idx], prepend=processed[0, idx])
                excessive = np.abs(velocity) > max_vel
                if excessive.any():
                    for i in np.where(excessive)[0]:
                        if i > 0:
                            target = processed[i-1, idx] + np.sign(velocity[i]) * max_vel
                            processed[i, idx] = target

        clamp_group(jaw_indices, self.constraints['max_velocity']['jaw'])
        clamp_group(mouth_indices, self.constraints['max_velocity']['mouth'])
        clamp_group(eye_indices, self.constraints['max_velocity']['eyes'])
        return processed

    def process(self, blendshapes):
        """Glavni deo za čišćenje sekvence."""
        processed = blendshapes.copy()
        processed = self.enforce_symmetry(processed)
        processed = self.clamp_extreme_velocity(processed)
        return np.clip(processed, 0, 1)


class ChampionshipInference:
    """
    Klasa za pokretanje modela i dobijanje finalnih rezultata iz audio fajla.
    """
    def __init__(self, model_path, device='cuda', fps=60):
        self.device = device
        self.fps = fps

        # Učitavanje modela uz ignorisanje VQ ključeva ako nisu potrebni
        checkpoint = torch.load(model_path, map_location=device, weights_only=False)
        self.model = ChampionshipLipSyncModel(
            audio_dim=1024, phoneme_embedding_dim=16, d_model=512,
            num_encoder_layers=4, num_heads=8, dim_feedforward=2048,
            num_blendshapes=52, codebook_size=512, gru_hidden=512, dropout=0.1
        ).to(device)
        self.model.load_state_dict(checkpoint['model_state_dict'], strict=False)
        self.model.eval()

        # Učitavanje Wav2Vec modela za zvuk
        self.feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("/content/drive/MyDrive/AI-SPEAK/srpski_wav2vec/model_0815_2ep_srKnjige")
        self.wav2vec = Wav2Vec2Model.from_pretrained("/content/drive/MyDrive/AI-SPEAK/srpski_wav2vec/model_0815_2ep_srKnjige").to(device)
        self.wav2vec.eval()
        for param in self.wav2vec.parameters(): param.requires_grad = False

        self.post_processor = MinimalPostProcessor(fps=self.fps)

    def extract_audio_features(self, audio_path, sr=16000):
        """Izvlačenje feature-a iz zvuka."""
        waveform_np, _ = librosa.load(audio_path, sr=sr)
        waveform = torch.from_numpy(waveform_np).unsqueeze(0)
        waveform = waveform / (waveform.abs().max() + 1e-8)
        with torch.no_grad():
            inputs = self.feature_extractor(waveform.squeeze().cpu().numpy(), sampling_rate=sr, return_tensors="pt")
            outputs = self.wav2vec(inputs.input_values.to(self.device), output_hidden_states=True)
            features = torch.stack(outputs.hidden_states[-4:]).mean(0).squeeze(0)
        features = (features - features.mean(dim=-1, keepdim=True)) / (features.std(dim=-1, keepdim=True) + 1e-8)

        return features.cpu().numpy()


    def _load_triphone_from_file(self, ph_path, num_frames, fps=60):
        phoneme_indices = np.zeros(num_frames, dtype=np.int64)
        with open(ph_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 3:
                    start, end, ph = float(parts[0]), float(parts[1]), parts[2].lower()
                    if ph in PHONEME_TO_IDX:
                        s_frame = int(start * fps)
                        e_frame = int(end * fps)
                        phoneme_indices[max(0, s_frame):min(num_frames, e_frame)] = PHONEME_TO_IDX[ph]

        prev_indices = np.concatenate([[phoneme_indices[0]], phoneme_indices[:-1]])
        next_indices = np.concatenate([phoneme_indices[1:], [phoneme_indices[-1]]])
        return prev_indices, phoneme_indices, next_indices


    @torch.no_grad()
    def predict(self, audio_path, phoneme_path=None, target_fps=60, apply_post_processing=True, auto_blink=True):
        """Dobijanje blendshape-ova i njihovo peglanje."""

        # 1. Obrada audia i interpolacija
        audio_features_np = self.extract_audio_features(audio_path)
        num_frames = int((len(audio_features_np) / 50) * target_fps)
        interp = interp1d(np.linspace(0, 1, len(audio_features_np)), audio_features_np, axis=0, kind='linear')
        audio_aligned = torch.from_numpy(interp(np.linspace(0, 1, num_frames))).float().unsqueeze(0).to(self.device)

        # 2. Učitavanje fonema
        if phoneme_path and os.path.exists(phoneme_path):
            # Pravi trifon iz .txt fajla
            prev_ph, curr_ph, next_ph = self._load_triphone_from_file(phoneme_path, num_frames, target_fps)
        else:
            # Fallback — energijska aproksimacija
            waveform_raw, _ = librosa.load(audio_path, sr=16000)
            energy = librosa.feature.rms(y=waveform_raw, frame_length=2048, hop_length=int(16000/target_fps))[0]
            energy = np.pad(energy, (0, max(0, num_frames - len(energy))))[:num_frames]
            curr_ph = np.where(energy > 0.02, 0, 31)
            prev_ph = np.concatenate([[curr_ph[0]], curr_ph[:-1]])
            next_ph = np.concatenate([curr_ph[1:], [curr_ph[-1]]])

        prev_tensor = torch.from_numpy(prev_ph).long().unsqueeze(0).to(self.device)
        curr_tensor = torch.from_numpy(curr_ph).long().unsqueeze(0).to(self.device)
        next_tensor = torch.from_numpy(next_ph).long().unsqueeze(0).to(self.device)

        # 3. Predikcija modela
        blendshapes, _, _ = self.model(
            audio_aligned,
            prev_tensor, curr_tensor, next_tensor,
            mask=torch.ones(1, num_frames, dtype=torch.bool).to(self.device)
        )

        blendshapes_np = blendshapes.squeeze(0).cpu().numpy()

        # 4. Savgol filter za glatkoću
        blendshapes_np[:, 13:15] = 0

        for i in range(blendshapes_np.shape[1]):
            window = 5 if i in [36, 24] else 9
            if len(blendshapes_np) > window:
                blendshapes_np[:, i] = savgol_filter(blendshapes_np[:, i], window, 2)

        if apply_post_processing:
            blendshapes_np = self.post_processor.process(blendshapes_np)

        # 5. Logika za automatsko treptanje
        if auto_blink:
            curr_f = random.randint(30, 90)
            while curr_f < num_frames - 20:
                duration = random.randint(10, 14)
                for i in range(duration):
                    if curr_f + i >= num_frames: break
                    t = i / duration
                    val = np.where(t < 0.25, t/0.25, 1.0 - (t-0.25)/0.75)
                    blendshapes_np[curr_f + i, 13] = float(val)
                    blendshapes_np[curr_f + i, 14] = float(val)
                curr_f += (duration + random.randint(2 * target_fps, 5 * target_fps))

        return {'blendshapes': blendshapes_np}



print("Inference system ready.")

In [ ]:
# ==============================================================================
# 10. SISTEM ZA VALIDACIJU I ANALIZU REZULTATA
# ==============================================================================

import matplotlib.image as mpimg

class ChampionshipValidator:
    """
    Glavna klasa za proveru modela na validacionim podacima (spk08 i spk14).
    Poredi ono što model predvidi sa originalnim podacima kroz razne metrike.
    """
    def __init__(self, model_path, base_path, speakers=['spk08', 'spk14'], device='cuda'):
        self.model_path = model_path
        self.base_path = base_path
        self.speakers = speakers
        self.device = device
        self.inference = ChampionshipInference(model_path, device, fps=60)

        # Koristimo istu logiku za podelu podataka kao u datasetu
        all_sentence_groups = []
        for speaker in self.speakers:
            is_new = speaker in ['spk03', 'spk04', 'spk05']
            spk_num = speaker.replace('spk', '')
            if is_new:
                csv_path = os.path.join(base_path, f'SPEAKER_{spk_num:0>2}',
                                        f'SPEAKER_{spk_num:0>2}_blendshapes_and_audio')
                if not os.path.exists(csv_path): continue
                f_ids = sorted([f.replace(f'{speaker}_', '').replace('.csv', '')
                                for f in os.listdir(csv_path)
                                if f.startswith(f'{speaker}_') and f.endswith('.csv')])
            else:
                csv_path = os.path.join(base_path, f'{speaker}_blendshapes', f'renamed_{speaker}')
                if not os.path.exists(csv_path): continue
                f_ids = sorted([f.replace(f'{speaker}_', '').replace('.csv', '')
                                for f in os.listdir(csv_path) if f.endswith('.csv')])
            for f_id in f_ids:
                all_sentence_groups.append((speaker, f_id))

        # Isti split kao u datasetu — 80/20
        SHARED_IDS    = {f'{i:03d}' for i in range(1, 31)}
        SHARED_VAL    = {f'{i:03d}' for i in range(21, 31)}

        # NOVO:
        old_shared_val = [(spk, fid) for spk, fid in all_sentence_groups
                          if spk in ['spk08', 'spk14'] and fid in SHARED_VAL]
        old_unique     = [(spk, fid) for spk, fid in all_sentence_groups
                          if spk in ['spk08', 'spk14'] and fid not in SHARED_IDS]

        new_shared_val = [(spk, fid) for spk, fid in all_sentence_groups
                          if spk in ['spk03', 'spk04', 'spk05'] and fid in SHARED_VAL]
        new_unique     = [(spk, fid) for spk, fid in all_sentence_groups
                          if spk in ['spk03', 'spk04', 'spk05'] and fid not in SHARED_IDS]

        random.seed(42)
        random.shuffle(old_unique)
        split_idx = int(len(old_unique) * 0.80)
        old_unique_val = old_unique[split_idx:]

        random.seed(42)
        random.shuffle(new_unique)
        split_idx2 = int(len(new_unique) * 0.80)
        new_unique_val = new_unique[split_idx2:]

        self.val_samples = old_shared_val + old_unique_val + new_shared_val + new_unique_val

        print(f"Validator ready. Testing on {len(self.val_samples)} unseen sentences.")

    def load_ground_truth(self, speaker, file_id):
        is_new = speaker in ['spk03', 'spk04', 'spk05']
        spk_num = speaker.replace('spk', '')
        if is_new:
            csv_path = os.path.join(self.base_path, f'SPEAKER_{spk_num:0>2}',
                                    f'SPEAKER_{spk_num:0>2}_blendshapes_and_audio',
                                    f'{speaker}_{file_id}.csv')
        else:
            csv_path = os.path.join(self.base_path, f'{speaker}_blendshapes',
                                    f'renamed_{speaker}', f'{speaker}_{file_id}.csv')
        try:
            df = pd.read_csv(csv_path, header=None)
            if df.shape[1] >= 52:
                blendshapes = df.iloc[:, :52].values.astype(np.float32)
            else:
                data = df.values.astype(np.float32)
                padding = np.zeros((data.shape[0], 52 - data.shape[1]), dtype=np.float32)
                blendshapes = np.concatenate([data, padding], axis=1)
            return blendshapes
        except Exception as e:
            print(f"Error loading ground truth for {speaker}_{file_id}: {e}")
            raise e

    def predict_single_file(self, speaker, file_id):
        is_new = speaker in ['spk03', 'spk04', 'spk05']
        spk_num = speaker.replace('spk', '')
        if is_new:
            audio = os.path.join(self.base_path, f'SPEAKER_{spk_num:0>2}',
                                f'SPEAKER_{spk_num:0>2}_blendshapes_and_audio',
                                f'FC_{file_id}.wav')
        else:
            audio = os.path.join(self.base_path, f'{speaker}_blendshapes',
                                f'renamed_{speaker}', f'{speaker}_{file_id}.wav')

        phoneme_path = os.path.join(self.base_path, 'labels_aligned', 'labels_aligned',
                                    'per_phoneme', f'{speaker}_{file_id}.txt')
        result = self.inference.predict(audio, phoneme_path=phoneme_path,
                                apply_post_processing=False, auto_blink=False)
        return result['blendshapes']

    def compute_metrics(self, pred, gt):
        """Računanje svih metrika korelacije, brzine, glatkoće i amplitude."""
        min_len = min(len(pred), len(gt))
        pred, gt = pred[:min_len], gt[:min_len]

        overall_corr, _ = pearsonr(pred.flatten(), gt.flatten())

        mouth_indices = list(range(28, 52))
        mouth_corr, _ = pearsonr(pred[:, mouth_indices].flatten(), gt[:, mouth_indices].flatten())
        jaw_corr, _ = pearsonr(pred[:, 24], gt[:, 24])

        pred_vel = np.diff(pred, axis=0)
        gt_vel = np.diff(gt, axis=0)
        vel_corr, _ = pearsonr(pred_vel.flatten(), gt_vel.flatten())

        pred_jitter = np.mean(np.abs(np.diff(pred_vel, axis=0)))
        gt_jitter = np.mean(np.abs(np.diff(gt_vel, axis=0)))

        pred_peaks = np.max(pred[:, [24, 28, 29]], axis=0)
        gt_peaks = np.max(gt[:, [24, 28, 29]], axis=0)
        amp_ratio = np.mean(pred_peaks / (gt_peaks + 1e-8))

        return {
            'correlation': overall_corr, 'mouth_correlation': mouth_corr, 'jaw_correlation': jaw_corr,
            'velocity_correlation': vel_corr, 'smoothness_ratio': pred_jitter / (gt_jitter + 1e-8),
            'amplitude_ratio': amp_ratio, 'mae': np.mean(np.abs(pred - gt)),
            'mouth_mae': np.mean(np.abs(pred[:, mouth_indices] - gt[:, mouth_indices]))
        }

    def visualize_prediction(self, speaker, file_id, save_path='validation_plots'):
        """Pravljenje vizuelnog prikaza predikcije kroz 6 različitih grafika."""
        os.makedirs(save_path, exist_ok=True)
        pred = self.predict_single_file(speaker, file_id)
        gt = self.load_ground_truth(speaker, file_id)
        min_len = min(len(pred), len(gt))
        pred, gt = pred[:min_len], gt[:min_len]
        m = self.compute_metrics(pred, gt)

        fig, axes = plt.subplots(2, 3, figsize=(24, 12))

        # Graf 1: Poređenje sinhronizacije bitnih kanala
        ax = axes[0, 0]
        idx_to_plot = [24, 28, 36, 37, 38]
        for i in idx_to_plot:
            ax.plot(gt[:, i], linestyle='--', alpha=0.4)
            ax.plot(pred[:, i], linewidth=2)
        ax.set_title(f"Synchronization Analysis (Corr: {m['correlation']:.3f})")


        # Graf 2: Mapa apsolutne greške
        ax = axes[0, 1]
        ax.imshow(np.abs(pred - gt).T, aspect='auto', cmap='hot')
        ax.set_title("Absolute Error Heatmap")

        # Graf 3: Korelacija po svakom od 52 blendshape-a
        ax = axes[0, 2]
        corrs = [pearsonr(pred[:, i], gt[:, i])[0] if np.std(gt[:, i]) > 1e-5 else 0 for i in range(52)]
        ax.bar(range(52), corrs)
        ax.set_title("Per-Channel Correlation")


        # Graf 4: Dinamika i brzina pokreta
        ax = axes[1, 0]
        ax.plot(np.linalg.norm(np.diff(gt, axis=0), axis=1), alpha=0.4, label='GT')
        ax.plot(np.linalg.norm(np.diff(pred, axis=0), axis=1), alpha=0.7, label='Pred')
        ax.set_title(f"Motion Dynamics (Vel Corr: {m['velocity_correlation']:.3f})")


        # Graf 5: Distribucija svih vrednosti
        ax = axes[1, 1]
        ax.hist(gt.flatten(), bins=50, alpha=0.4, label='GT', color='blue', density=True)
        ax.hist(pred.flatten(), bins=50, alpha=0.4, label='Pred', color='orange', density=True)
        ax.set_title("Value Distribution")


        # Graf 6: Detaljan prikaz vilice
        ax = axes[1, 2]
        ax.plot(gt[:, 24], linestyle='--', alpha=0.3)
        ax.plot(pred[:, 24], color='orange', alpha=0.6)
        ax.set_title(f"Jaw/Mouth Detail (MAE: {m['mouth_mae']:.4f})")

        plt.suptitle(f"Validation: {speaker}_{file_id}", fontsize=20, fontweight='bold')
        plt.savefig(os.path.join(save_path, f'{speaker}_{file_id}_analysis.png'), bbox_inches='tight')

        plt.show()
        plt.close()

    def generate_championship_report(self, num_visual_samples=5):
        """Pravljenje kompletnog izveštaja za sve uzorke i ispis statistike."""
        print("\n" + "-"*40 + "\nGLOBAL VALIDATION REPORT\n" + "-"*40)
        results = []

        for speaker, file_id in tqdm(self.val_samples):
            try:
                gt = self.load_ground_truth(speaker, file_id)
                pred = self.predict_single_file(speaker, file_id)
                m = self.compute_metrics(pred, gt)
                m['speaker'] = speaker
                m['file_id'] = file_id
                results.append(m)
            except Exception as e:
                pass

        if not results:
            print("Greška: Nijedan fajl nije uspešno validiran.")
            return pd.DataFrame()

        df = pd.DataFrame(results)
        if 'correlation' in df.columns:
            df = df.sort_values('correlation')

        # Prikaz 5 reprezentativnih primera (od najgoreg do najboljeg)
        n = len(df)
        indices = [0, n//4, n//2, 3*n//4, n-1]
        for idx in indices:
            row = df.iloc[idx]
            self.visualize_prediction(row['speaker'], row['file_id'])

        print("\n" + "="*120 + "\nSUMMARY STATISTICS\n" + "="*120)
        print(f"Total samples: {len(df)}")
        print(f"Speakers: {', '.join(self.speakers)}")
        print(f"\nMain metrics:")
        print(f"Mean correlation:     {df['correlation'].mean():.4f}")
        print(f"Mouth correlation:    {df['mouth_correlation'].mean():.4f}")
        print(f"Jaw correlation:      {df['jaw_correlation'].mean():.4f}")
        print(f"Velocity correlation: {df['velocity_correlation'].mean():.4f}")
        print(f"Smoothness ratio:     {df['smoothness_ratio'].mean():.4f}")
        print(f"MAE:                  {df['mae'].mean():.6f}")


        # Statistika po govorniku
        print(f"\nResults per speaker:")
        for speaker in self.speakers:
            speaker_df = df[df['speaker'] == speaker]
            print(f"\n{speaker.upper()}:")
            print(f"  Samples: {len(speaker_df)}")
            print(f"  Correlation: {speaker_df['correlation'].mean():.4f}")
            print(f"  Mouth Corr: {speaker_df['mouth_correlation'].mean():.4f}")
            print(f"  MAE: {speaker_df['mae'].mean():.6f}")

        print("="*120)
        return df


def analyze_per_blendshape(validator):
    """Detaljna analiza uspešnosti za svaki pojedinačni blendshape."""
    all_preds, all_gts = [], []

    for speaker, file_id in validator.val_samples:
        try:
            g = validator.load_ground_truth(speaker, file_id)
            p = validator.predict_single_file(speaker, file_id)
            min_l = min(len(p), len(g))
            all_preds.append(p[:min_l])
            all_gts.append(g[:min_l])
        except:
            continue

    if not all_preds:
        return pd.DataFrame()

    bp, bg = np.vstack(all_preds), np.vstack(all_gts)
    res = []

    for i, name in enumerate(BLENDSHAPE_NAMES):
        activity = np.std(bg[:, i])
        if activity > 1e-5 and np.std(bp[:, i]) > 1e-5:
            corr, _ = pearsonr(bp[:, i], bg[:, i])
        else:
            corr = 0.0

        res.append({
            'Index': i, 'Name': name, 'Correlation': corr,
            'MAE': np.mean(np.abs(bp[:, i] - bg[:, i])), 'Activity (GT)': activity
        })

    return pd.DataFrame(res).sort_values(by='Correlation', ascending=False)

print("Validation system loaded (all 5 speakers).")

In [ ]:
# ==============================================================================
# 11. POKRETANJE VALIDACIJE I ANALIZA REZULTATA
# ==============================================================================
from tqdm.auto import tqdm
inference = ChampionshipInference(
    model_path='/content/drive/MyDrive/AVATAR_CHECKPOINTS_5SPK_NODISCR/best_model.pt',
    device='cuda',
    fps=60
)

# Inicijalizacija validatora sa putanjom do najboljeg modela na Drive-u
validator = ChampionshipValidator(
    model_path='/content/drive/MyDrive/AVATAR_CHECKPOINTS_5SPK_NODISCR/best_model.pt',  # NOVI FOLDER
    base_path='/content/drive/MyDrive/AI-SPEAK',
    speakers=['spk08', 'spk14', 'spk03', 'spk04', 'spk05'],
    device='cuda'
)


# Generisanje globalnog izveštaja sa vizuelnim prikazima odabranih primera
validation_df = validator.generate_championship_report(num_visual_samples=5)

# Analiza uspešnosti za svaki pojedinačni blendshape kanal
bs_df = analyze_per_blendshape(validator)

# Ispis tabele sa rezultatima (korelacija, MAE i aktivnost originalnih podataka)
print(bs_df[['Index', 'Name', 'Correlation', 'MAE', 'Activity (GT)']])

In [ ]:
# ==============================================================================
# HELPER FUNCTIONS — Audio i fonemske putanje
# ==============================================================================

def get_audio_path(speaker, file_id):
    is_new = speaker in ['spk03', 'spk04', 'spk05']
    spk_num = speaker.replace('spk', '')
    if is_new:
        return os.path.join(BASE_DATA_PATH, f'SPEAKER_{spk_num:0>2}',
                            f'SPEAKER_{spk_num:0>2}_blendshapes_and_audio',
                            f'FC_{file_id}.wav')
    return os.path.join(BASE_DATA_PATH, f'{speaker}_blendshapes',
                        f'renamed_{speaker}', f'{speaker}_{file_id}.wav')

def get_phoneme_path(speaker, file_id):
    return os.path.join(BASE_DATA_PATH, 'labels_aligned', 'labels_aligned',
                        'per_phoneme', f'{speaker}_{file_id}.txt')

In [ ]:
# ==============================================================================
# TRAINING CURVES (Section 5.1)
# ==============================================================================
if 'trainer' not in dir() or not hasattr(trainer, 'history') or len(trainer.history['val_corr']) == 0:
    print("Training curves not available — run training first (Section 8).")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(1, len(trainer.history['val_corr']) + 1)

    axes[0].plot(epochs, trainer.history['train_corr'], label='Train', color='#3498DB', linewidth=2)
    axes[0].plot(epochs, trainer.history['val_corr'], label='Validation', color='#E74C3C', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=11)
    axes[0].set_ylabel('Pearson Correlation', fontsize=11)
    axes[0].set_title('Correlation over Training', fontsize=12, fontweight='bold')
    axes[0].legend(fontsize=10)
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, trainer.history['train_loss'], label='Train Loss', color='#3498DB', linewidth=2)
    axes[1].plot(epochs, trainer.history['val_loss'], label='Val Loss', color='#E74C3C', linewidth=2)
    axes[1].set_xlabel('Epoch', fontsize=11)
    axes[1].set_ylabel('Loss', fontsize=11)
    axes[1].set_title('Loss over Training', fontsize=12, fontweight='bold')
    axes[1].legend(fontsize=10)
    axes[1].grid(alpha=0.3)

    plt.suptitle('Training History', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ==============================================================================
# TRAINING TIME SUMMARY (Section 5.1)
# ==============================================================================
if 'trainer' not in dir() or not hasattr(trainer, 'history') or 'total_training_time_sec' not in trainer.history:
    print("Training time not available — run training first (Section 8).")
else:
    t = trainer.history['total_training_time_sec']
    hours   = int(t // 3600)
    minutes = int((t % 3600) // 60)
    seconds = int(t % 60)
    epochs  = trainer.history['epochs_trained']

    print("="*50)
    print("TRAINING TIME SUMMARY")
    print("="*50)
    print(f"  Total time:            {hours}h {minutes}m {seconds}s")
    print(f"  Total seconds:         {t:.1f}s")
    print(f"  Epochs trained:        {epochs}")
    print(f"  Avg time per epoch:    {t/epochs:.1f}s")
    print(f"  Hardware:              T4 GPU (Google Colab)")
    print(f"  Batch size:            8 x 8 accumulation = 64 effective")
    print(f"  Training sentences:    634")
    print(f"  Validation sentences:  174")
    print("="*50)

In [ ]:
# ==============================================================================
# PER-SPEAKER RESULTS TABLE (Section 5.3)
# ==============================================================================

print("\n" + "="*80)
print("PER-SPEAKER VALIDATION RESULTS")
print("="*80)
print(f"{'Speaker':<12} {'N':<6} {'Corr':<8} {'Mouth Corr':<12} {'Jaw Corr':<10} {'Vel Corr':<10} {'MAE':<8}")
print("-"*80)

for spk in ['spk08', 'spk14', 'spk03', 'spk04', 'spk05']:
    spk_df = validation_df[validation_df['speaker'] == spk]
    if len(spk_df) == 0: continue
    print(f"{spk:<12} {len(spk_df):<6} "
          f"{spk_df['correlation'].mean():.4f}   "
          f"{spk_df['mouth_correlation'].mean():.4f}       "
          f"{spk_df['jaw_correlation'].mean():.4f}     "
          f"{spk_df['velocity_correlation'].mean():.4f}     "
          f"{spk_df['mae'].mean():.4f}")

print("-"*80)
print(f"{'ALL':<12} {len(validation_df):<6} "
      f"{validation_df['correlation'].mean():.4f}   "
      f"{validation_df['mouth_correlation'].mean():.4f}       "
      f"{validation_df['jaw_correlation'].mean():.4f}     "
      f"{validation_df['velocity_correlation'].mean():.4f}     "
      f"{validation_df['mae'].mean():.4f}")

In [ ]:
# ==============================================================================
# CORRELATION PER BLENDSHAPE - FIGURE FOR PAPER (Section 5.3)
# ==============================================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, ax = plt.subplots(figsize=(16, 5))

colors = []
for name in bs_df['Name']:
    if 'jaw' in name.lower(): colors.append('#E74C3C')
    elif 'mouth' in name.lower(): colors.append('#3498DB')
    elif 'brow' in name.lower(): colors.append('#2ECC71')
    elif 'eye' in name.lower(): colors.append('#9B59B6')
    else: colors.append('#95A5A6')

bars = ax.bar(range(len(bs_df)), bs_df['Correlation'], color=colors, edgecolor='white', linewidth=0.5)
ax.axhline(y=bs_df['Correlation'].mean(), color='black', linestyle='--',
           linewidth=1.5, label=f"Mean = {bs_df['Correlation'].mean():.3f}")
ax.set_xticks(range(len(bs_df)))
ax.set_xticklabels(bs_df['Name'], rotation=90, fontsize=7)
ax.set_ylabel('Pearson Correlation', fontsize=11)
ax.set_title('Per-Blendshape Correlation (All Speakers)', fontsize=12, fontweight='bold')
ax.set_ylim(-0.1, 1.05)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

legend_elements = [
    mpatches.Patch(color='#E74C3C', label='Jaw'),
    mpatches.Patch(color='#3498DB', label='Mouth'),
    mpatches.Patch(color='#2ECC71', label='Brow'),
    mpatches.Patch(color='#9B59B6', label='Eye'),
    mpatches.Patch(color='#95A5A6', label='Other'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('blendshape_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================================================
# 12. POSTPROCESSING
# ==============================================================================

from scipy.ndimage import uniform_filter1d, gaussian_filter1d

def process_lip_sync(result, output_path, audio_path_ref=None):
    """
    Minimalni postprocessing: samo automatsko treptanje.
    """
    blendshapes = result['blendshapes'].copy()
    num_frames = blendshapes.shape[0]

    # Automatsko treptanje
    blendshapes[:, 13] = 0.0
    blendshapes[:, 14] = 0.0

    curr_f = random.randint(40, 80)
    while curr_f < num_frames - 20:
        duration = random.randint(10, 14)
        for k in range(duration):
            if curr_f + k >= num_frames:
                break
            t = k / duration
            val = t / 0.25 if t < 0.25 else 1.0 - (t - 0.25) / 0.75
            blendshapes[curr_f + k, 13] = float(val)
            blendshapes[curr_f + k, 14] = float(val)
        curr_f += (duration + random.randint(120, 300))

    blendshapes = np.clip(blendshapes, 0, 1)
    pd.DataFrame(blendshapes).to_csv(output_path, header=False, index=False, float_format='%.6f')
    print(f"Export complete: {output_path}")

In [ ]:
# ==============================================================================
# 13. GENERISANJE VALIDACIONIH CSV FAJLOVA (ceo validacioni skup)
# ==============================================================================

import os
import pandas as pd
from tqdm.auto import tqdm

OUTPUT_VAL_DIR = '/content/drive/MyDrive/AVATAR_OUTPUT_5SPK_NODISCR/validacioni_test'
DIR_RAW  = os.path.join(OUTPUT_VAL_DIR, 'raw')
DIR_POST = os.path.join(OUTPUT_VAL_DIR, 'postprocessed')
os.makedirs(DIR_RAW, exist_ok=True)
os.makedirs(DIR_POST, exist_ok=True)

print(f"Generating CSV for all {len(validator.val_samples)} validation samples...")

for speaker, file_id in tqdm(validator.val_samples, desc="Generating CSVs"):
    audio_path   = get_audio_path(speaker, file_id)
    phoneme_path = get_phoneme_path(speaker, file_id)

    if not os.path.exists(audio_path):
        print(f"  ⚠️ Audio not found: {audio_path}")
        continue

    try:
        # Bez postprocessinga
        rezultat = inference.predict(audio_path, phoneme_path=phoneme_path,
                                     apply_post_processing=False, auto_blink=False)
        raw_bs = rezultat['blendshapes'].copy()
        raw_path = os.path.join(DIR_RAW, f'{speaker}_{file_id}.csv')
        pd.DataFrame(raw_bs).to_csv(raw_path, header=False, index=False, float_format='%.6f')

        # Sa postprocessingom
        post_path = os.path.join(DIR_POST, f'{speaker}_{file_id}.csv')
        process_lip_sync(rezultat, post_path)

    except Exception as e:
        print(f"  ⚠️ Error on {speaker}_{file_id}: {e}")

print(f"\nDone. Saved {len(validator.val_samples)} files to:")
print(f"  RAW:  {DIR_RAW}")
print(f"  POST: {DIR_POST}")

In [ ]:
# ==============================================================================
# RTF MEASUREMENT (Section 5.1 / 5.2)
# ==============================================================================
print("Measuring inference time on full validation set...")
print("="*65)
print(f"{'File':<25} {'Duration (s)':>12} {'Infer (s)':>10} {'RTF':>7}")
print("="*65)

rtf_rows = []

for speaker, file_id in tqdm(validator.val_samples, desc="RTF measurement"):
    audio_path   = get_audio_path(speaker, file_id)
    phoneme_path = get_phoneme_path(speaker, file_id)

    if not os.path.exists(audio_path):
        continue

    try:
        audio_duration = librosa.get_duration(path=audio_path)
        start = time.perf_counter()
        res = inference.predict(audio_path, phoneme_path=phoneme_path,
                        apply_post_processing=False, auto_blink=False)
        process_lip_sync(res, '/tmp/rtf_tmp.csv')
        infer_time = time.perf_counter() - start
        rtf = infer_time / audio_duration

        rtf_rows.append({
            'Speaker': speaker,
            'File': file_id,
            'Duration (s)': round(audio_duration, 3),
            'Inference (s)': round(infer_time, 4),
            'RTF': round(rtf, 4),
            'Lookahead': 'full_file',
            'Mode': 'offline'
        })

        print(f"{speaker}_{file_id:<15} {audio_duration:>12.3f} {infer_time:>10.4f} {rtf:>7.4f}")

    except Exception as e:
        print(f"Error on {speaker}_{file_id}: {e}")

print("="*65)

# DataFrame sa rezultatima
df_rtf = pd.DataFrame(rtf_rows)

# Summary statistika
print("\n" + "="*65)
print("RTF SUMMARY STATISTICS")
print("="*65)
print(f"{'Total files measured:':<30} {len(df_rtf)}")
print(f"{'Mean audio duration (s):':<30} {df_rtf['Duration (s)'].mean():.3f}")
print(f"{'Mean inference time (s):':<30} {df_rtf['Inference (s)'].mean():.4f}")
print(f"{'Mean RTF:':<30} {df_rtf['RTF'].mean():.4f}")
print(f"{'Median RTF:':<30} {df_rtf['RTF'].median():.4f}")
print(f"{'Min RTF:':<30} {df_rtf['RTF'].min():.4f}")
print(f"{'Max RTF:':<30} {df_rtf['RTF'].max():.4f}")
print(f"{'Std RTF:':<30} {df_rtf['RTF'].std():.4f}")
print(f"{'Real-time capable (RTF<1):':<30} {(df_rtf['RTF'] < 1).sum()}/{len(df_rtf)}")
print(f"{'Lookahead:':<30} full_file (offline mode)")
print(f"{'System lookahead (ms):':<30} {LOOKAHEAD_MS} ms (defined but not enforced)")
print("="*65)

# Per-speaker RTF statistika
print("\nRTF PER SPEAKER:")
print("-"*55)
print(f"{'Speaker':<12} {'N':<6} {'Mean RTF':<12} {'Min RTF':<10} {'Max RTF':<10}")
print("-"*55)
for spk in ['spk08', 'spk14', 'spk03', 'spk04', 'spk05']:
    spk_rtf = df_rtf[df_rtf['Speaker'] == spk]
    if len(spk_rtf) == 0: continue
    print(f"{spk:<12} {len(spk_rtf):<6} "
          f"{spk_rtf['RTF'].mean():<12.4f} "
          f"{spk_rtf['RTF'].min():<10.4f} "
          f"{spk_rtf['RTF'].max():<10.4f}")
print("-"*55)
print(f"{'ALL':<12} {len(df_rtf):<6} "
      f"{df_rtf['RTF'].mean():<12.4f} "
      f"{df_rtf['RTF'].min():<10.4f} "
      f"{df_rtf['RTF'].max():<10.4f}")

# Grafik distribucije RTF vrednosti
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_rtf['RTF'], bins=20, color='#3498DB', edgecolor='white', linewidth=0.5)
axes[0].axvline(df_rtf['RTF'].mean(), color='#E74C3C', linestyle='--',
                linewidth=2, label=f"Mean = {df_rtf['RTF'].mean():.3f}")
axes[0].axvline(1.0, color='black', linestyle=':', linewidth=1.5, label='RTF = 1.0 (real-time)')
axes[0].set_xlabel('RTF', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('RTF Distribution (Full Validation Set)', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(axis='y', alpha=0.3)

axes[1].scatter(df_rtf['Duration (s)'], df_rtf['RTF'],
                color='#3498DB', alpha=0.6, edgecolors='white', linewidth=0.5)
axes[1].axhline(1.0, color='black', linestyle=':', linewidth=1.5, label='RTF = 1.0')
axes[1].axhline(df_rtf['RTF'].mean(), color='#E74C3C', linestyle='--',
                linewidth=2, label=f"Mean = {df_rtf['RTF'].mean():.3f}")
axes[1].set_xlabel('Audio Duration (s)', fontsize=11)
axes[1].set_ylabel('RTF', fontsize=11)
axes[1].set_title('RTF vs Audio Duration', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.suptitle('Inference Speed Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('rtf_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nNote: Mode = offline — model processes entire utterance at once.")
print(f"For real-time constraint, lookahead would be limited to {LOOKAHEAD_MS}ms.")

In [ ]:
# ==============================================================================
# PRE vs POST PROCESSING VISUALIZATION (Section 4)
# ==============================================================================
import librosa

spk, fid = validator.val_samples[0]
audio_path   = get_audio_path(spk, fid)
phoneme_path = get_phoneme_path(spk, fid)

# RAW — bez ikakvog postprocessinga
res_raw = inference.predict(audio_path, phoneme_path=phoneme_path,
                             apply_post_processing=False, auto_blink=False)
raw_bs = res_raw['blendshapes'].copy()

# POST — kroz process_lip_sync
tmp_path = '/tmp/viz_post.csv'
process_lip_sync(res_raw, tmp_path)
post_bs = pd.read_csv(tmp_path, header=None).values.astype(np.float32)

min_len = min(len(raw_bs), len(post_bs))
t = np.linspace(0, min_len/60, min_len)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(t, raw_bs[:min_len, 24],  label='Jaw (raw)',         color='#E74C3C', alpha=0.7, linewidth=1.5)
axes[0].plot(t, post_bs[:min_len, 24], label='Jaw (post)',        color='#C0392B', linewidth=2)
axes[0].plot(t, raw_bs[:min_len, 36],  label='mouthClose (raw)',  color='#3498DB', alpha=0.7, linewidth=1.5)
axes[0].plot(t, post_bs[:min_len, 36], label='mouthClose (post)', color='#1A5276', linewidth=2)
axes[0].set_ylabel('Blendshape Value', fontsize=11)
axes[0].set_title(f'Pre vs Post Processing — {spk}_{fid}', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

y, sr = librosa.load(audio_path, sr=16000)
energy = librosa.feature.rms(y=y, frame_length=2048, hop_length=int(16000/60))[0]
energy = energy / (energy.max() + 1e-8)
t_e = np.linspace(0, min_len/60, min(len(energy), min_len))

axes[1].fill_between(t_e, 0, energy[:min_len], color='#95A5A6', alpha=0.4, label='Audio energy')
axes[1].plot(t, post_bs[:min_len, 24], label='Jaw (post)', color='#C0392B', linewidth=2)
axes[1].plot(t, post_bs[:min_len, 36], label='mouthClose (post)', color='#1A5276', linewidth=2)
axes[1].set_xlabel('Time (s)', fontsize=11)
axes[1].set_ylabel('Value', fontsize=11)
axes[1].set_title('Audio Energy vs Final Blendshapes', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('postprocessing_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================================================
# CORRELATION DISTRIBUTION PER SPEAKER (Section 5.3)
# ==============================================================================
import seaborn as sns

fig, ax = plt.subplots(figsize=(10, 5))
speakers_order = ['spk08', 'spk14', 'spk03', 'spk04', 'spk05']
palette = ['#3498DB', '#E74C3C', '#2ECC71', '#F39C12', '#9B59B6']

sns.boxplot(data=validation_df, x='speaker', y='correlation',
            order=speakers_order, palette=palette, ax=ax)
ax.axhline(validation_df['correlation'].mean(), color='black',
           linestyle='--', linewidth=1.5, label=f"Mean = {validation_df['correlation'].mean():.3f}")
ax.set_xlabel('Speaker', fontsize=11)
ax.set_ylabel('Pearson Correlation', fontsize=11)
ax.set_title('Correlation Distribution per Speaker', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('correlation_per_speaker.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================================================
# ABLATION STUDY TABLE (Section 5.2) — popuniti posle eksperimenata
# ==============================================================================

ablation_data = {
    'Model Variant': [
        'Full model (audio + triphone + postproc)',
        'Audio only (no phoneme)',
        'No postprocessing',
        'No discriminator',
        'Offline (no lookahead limit)',
    ],
    'Corr': ['—', '—', '—', '—', '—'],
    'Mouth Corr': ['—', '—', '—', '—', '—'],
    'MAE': ['—', '—', '—', '—', '—'],
    'RTF': ['—', '—', '—', '—', '—'],
}

df_ablation = pd.DataFrame(ablation_data)
print("\nABLATION STUDY (to be filled after experiments):")
print(df_ablation.to_string(index=False))

In [ ]:
# ==============================================================================
# ABLATION STUDY — RAW PREDICTION (no postprocessing, no filter)
# ==============================================================================

@torch.no_grad()
def predict_raw(inference_obj, audio_path, phoneme_path=None, target_fps=60):
    """
    Čista predikcija modela — bez savgol filtera, bez MinimalPostProcessor,
    bez treptanja. Samo forward pass modela.
    """
    audio_features_np = inference_obj.extract_audio_features(audio_path)
    num_frames = int((len(audio_features_np) / 50) * target_fps)
    interp_fn = interp1d(
        np.linspace(0, 1, len(audio_features_np)),
        audio_features_np, axis=0, kind='linear'
    )
    audio_aligned = torch.from_numpy(
        interp_fn(np.linspace(0, 1, num_frames))
    ).float().unsqueeze(0).to(inference_obj.device)

    if phoneme_path and os.path.exists(phoneme_path):
        prev_ph, curr_ph, next_ph = inference_obj._load_triphone_from_file(
            phoneme_path, num_frames, target_fps)
    else:
        sil_idx = PHONEME_TO_IDX['sil']
        curr_ph = np.full(num_frames, sil_idx, dtype=np.int64)
        prev_ph = curr_ph.copy()
        next_ph = curr_ph.copy()

    prev_t = torch.from_numpy(prev_ph).long().unsqueeze(0).to(inference_obj.device)
    curr_t = torch.from_numpy(curr_ph).long().unsqueeze(0).to(inference_obj.device)
    next_t = torch.from_numpy(next_ph).long().unsqueeze(0).to(inference_obj.device)

    blendshapes, _, _ = inference_obj.model(
        audio_aligned, prev_t, curr_t, next_t,
        mask=torch.ones(1, num_frames, dtype=torch.bool).to(inference_obj.device)
    )

    return np.clip(blendshapes.squeeze(0).cpu().numpy(), 0, 1)


def compute_metrics_raw(pred, gt):
    min_len = min(len(pred), len(gt))
    pred, gt = pred[:min_len], gt[:min_len]
    mouth_indices = list(range(28, 52))

    overall_corr, _ = pearsonr(pred.flatten(), gt.flatten())
    mouth_corr,   _ = pearsonr(pred[:, mouth_indices].flatten(), gt[:, mouth_indices].flatten())
    jaw_corr,     _ = pearsonr(pred[:, 24], gt[:, 24])
    vel_corr,     _ = pearsonr(
        np.diff(pred, axis=0).flatten(),
        np.diff(gt,   axis=0).flatten()
    )
    mae       = np.mean(np.abs(pred - gt))
    mouth_mae = np.mean(np.abs(pred[:, mouth_indices] - gt[:, mouth_indices]))

    return {
        'correlation':       overall_corr,
        'mouth_correlation': mouth_corr,
        'jaw_correlation':   jaw_corr,
        'velocity_correlation': vel_corr,
        'mae':       mae,
        'mouth_mae': mouth_mae,
    }


print("Running ablation: raw prediction (no filter, no postproc)...")
ablation_rows = []

for speaker, file_id in tqdm(validator.val_samples, desc="Ablation raw"):
    audio_path   = get_audio_path(speaker, file_id)
    phoneme_path = get_phoneme_path(speaker, file_id)
    if not os.path.exists(audio_path):
        continue
    try:
        gt   = validator.load_ground_truth(speaker, file_id)
        pred = predict_raw(inference, audio_path, phoneme_path)
        m    = compute_metrics_raw(pred, gt)
        m['speaker'] = speaker
        m['file_id'] = file_id
        ablation_rows.append(m)
    except Exception as e:
        print(f"Error {speaker}_{file_id}: {e}")

df_abl = pd.DataFrame(ablation_rows)

print("\n" + "="*80)
print("ABLATION — RAW PREDICTION (no filter, no postprocessing)")
print("="*80)
print(f"{'Speaker':<12} {'N':<6} {'Corr':<8} {'Mouth':<8} {'Jaw':<8} {'Vel':<8} {'MAE':<10} {'MouthMAE'}")
print("-"*80)
for spk in ['spk08', 'spk14', 'spk03', 'spk04', 'spk05']:
    s = df_abl[df_abl['speaker'] == spk]
    if len(s) == 0: continue
    print(f"{spk:<12} {len(s):<6} "
          f"{s['correlation'].mean():.4f}   "
          f"{s['mouth_correlation'].mean():.4f}   "
          f"{s['jaw_correlation'].mean():.4f}   "
          f"{s['velocity_correlation'].mean():.4f}   "
          f"{s['mae'].mean():.6f}   "
          f"{s['mouth_mae'].mean():.6f}")
print("-"*80)
print(f"{'ALL':<12} {len(df_abl):<6} "
      f"{df_abl['correlation'].mean():.4f}   "
      f"{df_abl['mouth_correlation'].mean():.4f}   "
      f"{df_abl['jaw_correlation'].mean():.4f}   "
      f"{df_abl['velocity_correlation'].mean():.4f}   "
      f"{df_abl['mae'].mean():.6f}   "
      f"{df_abl['mouth_mae'].mean():.6f}")
print("="*80)

In [ ]:
# ==============================================================================
# RTF MEASUREMENT — NODISCR RAW
# ==============================================================================
print("Measuring RTF for NODISCR raw prediction...")
print("="*65)
print(f"{'File':<25} {'Duration (s)':>12} {'Infer (s)':>10} {'RTF':>7}")
print("="*65)

rtf_rows_nodiscr = []

for speaker, file_id in tqdm(validator.val_samples, desc="RTF nodiscr raw"):
    audio_path   = get_audio_path(speaker, file_id)
    phoneme_path = get_phoneme_path(speaker, file_id)

    if not os.path.exists(audio_path):
        continue

    try:
        audio_duration = librosa.get_duration(path=audio_path)
        start = time.perf_counter()
        predict_raw(inference, audio_path, phoneme_path)
        infer_time = time.perf_counter() - start
        rtf = infer_time / audio_duration

        rtf_rows_nodiscr.append({
            'Speaker': speaker,
            'File': file_id,
            'Duration (s)': round(audio_duration, 3),
            'Inference (s)': round(infer_time, 4),
            'RTF': round(rtf, 4),
            'Mode': 'nodiscr_raw'
        })

        print(f"{speaker}_{file_id:<15} {audio_duration:>12.3f} {infer_time:>10.4f} {rtf:>7.4f}")

    except Exception as e:
        print(f"Error on {speaker}_{file_id}: {e}")

print("="*65)

df_rtf_nodiscr = pd.DataFrame(rtf_rows_nodiscr)

print("\n" + "="*65)
print("RTF SUMMARY — NODISCR RAW")
print("="*65)
print(f"{'Total files:':<30} {len(df_rtf_nodiscr)}")
print(f"{'Mean RTF:':<30} {df_rtf_nodiscr['RTF'].mean():.4f}")
print(f"{'Median RTF:':<30} {df_rtf_nodiscr['RTF'].median():.4f}")
print(f"{'Min RTF:':<30} {df_rtf_nodiscr['RTF'].min():.4f}")
print(f"{'Max RTF:':<30} {df_rtf_nodiscr['RTF'].max():.4f}")
print(f"{'Real-time capable (RTF<1):':<30} {(df_rtf_nodiscr['RTF'] < 1).sum()}/{len(df_rtf_nodiscr)}")
print("="*65)

print("\nRTF PER SPEAKER:")
print("-"*55)
print(f"{'Speaker':<12} {'N':<6} {'Mean RTF':<12} {'Min RTF':<10} {'Max RTF':<10}")
print("-"*55)
for spk in ['spk08', 'spk14', 'spk03', 'spk04', 'spk05']:
    s = df_rtf_nodiscr[df_rtf_nodiscr['Speaker'] == spk]
    if len(s) == 0: continue
    print(f"{spk:<12} {len(s):<6} "
          f"{s['RTF'].mean():<12.4f} "
          f"{s['RTF'].min():<10.4f} "
          f"{s['RTF'].max():<10.4f}")
print("-"*55)
print(f"{'ALL':<12} {len(df_rtf_nodiscr):<6} "
      f"{df_rtf_nodiscr['RTF'].mean():<12.4f} "
      f"{df_rtf_nodiscr['RTF'].min():<10.4f} "
      f"{df_rtf_nodiscr['RTF'].max():<10.4f}")